# Notebook 06 – Model Evaluation

Dieses Notebook führt die finale Evaluation der traineirten RL-Agenten durch.

Dafür wird sich an die folgenden Regeln gehalten:
- Finale Evaluation wird auf 50 Episoden pro Run gesetzt
- Wichtig: Alte 200-Episoden-Evaluations-Runs wurden manuell archiviert und nicht mehr in die finale Auswertung geladen (eine genauere Erklärung dazu findet sich weiter unten)
- A2C und DQN werden in Env1 deterministisch und stochastisch evaluiert (Begründung erfolgt weiter unten)
- PPO wird separat über Unity ML-Agents Inference behandelt
- Transfer wird nur mit den besten Modellen aus der Evaluation des Env1 durchgeführt (genauer Begründung erfolgt ebenfalls weiter unten)

---

### 1. Imports und Projektpfade

Diese Zelle lädt die benötigten Bibliotheken und setzt die zentralen Projektpfade für Modelle, Logs und Evaluationsergebnisse.

In [3]:
from __future__ import annotations

import sys
import os
import json
import time
import random
import warnings
from pathlib import Path
from typing import Any, Optional

import numpy as np
import pandas as pd
import torch
import gymnasium as gym
from gymnasium import spaces
import shutil

from stable_baselines3 import A2C, DQN

from mlagents_envs.environment import UnityEnvironment
from mlagents_envs.envs.unity_gym_env import UnityToGymWrapper

warnings.filterwarnings("ignore")

# Projektwurzel wie in Notebook 05 setzen
PROJECT_ROOT_NOTEBOOK = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT_NOTEBOOK))

from lib_py.paths import (
    PROJECT_ROOT,
    PROJECT_NAME,
    MAZE_AGENT_DIR,
    TRAINING_DIR,
    MODEL_DIR,
    LOG_DIR,
    ensure_project_dirs,
    show_project_path,
)

# Working Directory auf Projektwurzel setzen
os.chdir(PROJECT_ROOT)

# Zentrale Projektordner
ensure_project_dirs()

CONFIG_DIR = TRAINING_DIR / "configs"
TUNING_DIR = TRAINING_DIR / "tuning"
FINAL_MODEL_DIR = MODEL_DIR / "final"
FINAL_LOG_DIR = LOG_DIR / "final"
RUN_SUMMARY_DIR = TRAINING_DIR / "run_summaries"
RECORDING_DIR = TRAINING_DIR / "recordings"

EVALUATION_DIR = TRAINING_DIR / "evaluation_results"
EVALUATION_RAW_DIR = EVALUATION_DIR / "raw"
EVALUATION_TABLE_DIR = EVALUATION_DIR / "tables"

CONFIG_DIR.mkdir(parents=True, exist_ok=True)
TUNING_DIR.mkdir(parents=True, exist_ok=True)
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)
FINAL_LOG_DIR.mkdir(parents=True, exist_ok=True)
RUN_SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
RECORDING_DIR.mkdir(parents=True, exist_ok=True)
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)
EVALUATION_RAW_DIR.mkdir(parents=True, exist_ok=True)
EVALUATION_TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("Imports funktionieren.")
print("Project Name:", PROJECT_NAME)
print("Project Root:", show_project_path(PROJECT_ROOT))
print("Current Working Directory:", show_project_path(Path.cwd()))
print("Final Model Dir:", show_project_path(FINAL_MODEL_DIR))
print("Final Log Dir:", show_project_path(FINAL_LOG_DIR))
print("Run Summary Dir:", show_project_path(RUN_SUMMARY_DIR))
print("Evaluation Raw Dir:", show_project_path(EVALUATION_RAW_DIR))
print("Evaluation Table Dir:", show_project_path(EVALUATION_TABLE_DIR))

Imports funktionieren.
Project Name: rl_navigation_project
Project Root: rl_navigation_project/.
Current Working Directory: rl_navigation_project/.
Final Model Dir: rl_navigation_project/training/models/final
Final Log Dir: rl_navigation_project/training/logs/final
Run Summary Dir: rl_navigation_project/training/run_summaries
Evaluation Raw Dir: rl_navigation_project/training/evaluation_results/raw
Evaluation Table Dir: rl_navigation_project/training/evaluation_results/tables


---

### 2. Evaluation-Konfiguration

Diese Zelle legt die zentralen Evaluationseinstellungen fest. Geplant sind 200 Episoden pro Modell und Umgebung.

In [4]:
EVAL_EPISODES = 50
EVAL_MAX_STEPS_PER_EPISODE = 5000

TRAIN_ENV_NAME = "Env1"

EVAL_ENVIRONMENTS = [
    "Env1",
    "Env2",
    "Env3",
]

print("Evaluation Episodes:", EVAL_EPISODES)
print("Max Steps pro Episode:", EVAL_MAX_STEPS_PER_EPISODE)
print("Trainingsumgebung:", TRAIN_ENV_NAME)
print("Evaluation Environments:", EVAL_ENVIRONMENTS)


Evaluation Episodes: 50
Max Steps pro Episode: 5000
Trainingsumgebung: Env1
Evaluation Environments: ['Env1', 'Env2', 'Env3']


### 2.5 NumPy-Kompatibilität für gespeicherte SB3-Modelle

Diese Zelle verhindert Ladefehler bei SB3-Modellen, die mit einer anderen Numpy-Version gespeichert wurden

In [5]:
try:
    import numpy.core as numpy_core
    import numpy.core.multiarray as numpy_core_multiarray
    import numpy.core.numeric as numpy_core_numeric
    import numpy.core._multiarray_umath as numpy_core_multiarray_umath

    sys.modules.setdefault("numpy._core", numpy_core)
    sys.modules.setdefault("numpy._core.multiarray", numpy_core_multiarray)
    sys.modules.setdefault("numpy._core.numeric", numpy_core_numeric)
    sys.modules.setdefault("numpy._core._multiarray_umath", numpy_core_multiarray_umath)

    print("NumPy compatibility aliases gesetzt.")
    print("NumPy Version:", np.__version__)

except Exception as exc:
    print("NumPy compatibility aliases konnten nicht gesetzt werden:")
    print(type(exc).__name__, exc)

NumPy compatibility aliases gesetzt.
NumPy Version: 1.23.5


### 3. Globale Seeds setzen

Diese Funktion setzt Python-, NumPy- und Random-Seeds, damit die Evaluation reproduzierbarer wird

In [6]:
def set_global_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)

    try:
        torch.manual_seed(seed)

        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except Exception:
        pass

    print(f"Seeds gesetzt auf: {seed}")

---

### 4. Unity-Environment erstellen

Diese Funktion verbindet Python mit der aktuell geöffneten Unity-Szene und wrapped sie als Gym-Environment.  

Für A2C und DQN gilt:
- Unity-Szene öffnen
- MazeAgent Behavior Type auf `Default`
- kein ONNX-Modell eintragen
- Python-Zelle starten
- anschließend in Unity Play drücken

In [7]:
def make_unity_env(seed: int, worker_id: int = 0):
    set_global_seeds(seed)

    print("Starte Verbindung zu Unity. Jetzt in Unity Play drücken...")

    unity_env = UnityEnvironment(
        file_name=None,
        seed=seed,
        worker_id=worker_id,
        no_graphics=False,
    )

    env = UnityToGymWrapper(
        unity_env,
        uint8_visual=False,
        flatten_branched=True,
        allow_multiple_obs=False,
    )

    print("Unity Environment verbunden.")
    print("Observation Space:", env.observation_space)
    print("Action Space:", env.action_space)

    return env

---

### 5. Hilfsfunktionen für SB3-Umgebungen

Diese Funktionen vereinheitlichen Reset, Step und Done-Handling, weil Unity-/SB3-Wrapper teilweise leicht unterschiedliche Rückgabeformate verwenden.

In [8]:
def reset_env(env):
    reset_result = env.reset()

    if isinstance(reset_result, tuple) and len(reset_result) == 2:
        obs, info = reset_result
        return obs

    return reset_result


def step_env(env, action):
    step_result = env.step(action)

    if isinstance(step_result, tuple) and len(step_result) == 5:
        obs, reward, terminated, truncated, info = step_result
        done = terminated or truncated
        return obs, reward, done, info

    if isinstance(step_result, tuple) and len(step_result) == 4:
        obs, reward, done, info = step_result
        return obs, reward, done, info

    raise ValueError(f"Unbekanntes env.step() Rückgabeformat: {type(step_result)}")


def to_scalar(value):
    if isinstance(value, (list, tuple, np.ndarray)):
        arr = np.array(value).flatten()

        if len(arr) == 0:
            return None

        return arr[0].item()

    return value


def done_to_bool(done) -> bool:
    return bool(to_scalar(done))


def reward_to_float(reward) -> float:
    return float(to_scalar(reward))


def info_to_dict(info) -> dict:
    if isinstance(info, list) and len(info) > 0:
        if isinstance(info[0], dict):
            return info[0]
        return {"info": str(info[0])}

    if isinstance(info, dict):
        return info

    return {"info": str(info)}

---

### 6. Run-Summaries laden

Diese Zelle lädt die bereinigten Run-Summary-Dateien. Für DQN wird die bereinigte DQN-v1-Datei verwendet. 

In [9]:
def load_csv_if_exists(path: Path, label: str) -> pd.DataFrame:
    if path.exists():
        df = pd.read_csv(path)
        print(f"{label} geladen:", show_project_path(path), f"({len(df)} Zeilen)")
        return df

    print(f"{label} nicht gefunden:", show_project_path(path))
    return pd.DataFrame()


a2c_runs_df = load_csv_if_exists(
    RUN_SUMMARY_DIR / "a2c_final_training_runs.csv",
    "A2C Runs"
)

dqn_runs_df = load_csv_if_exists(
    RUN_SUMMARY_DIR / "dqn_v1_final_evaluation_runs.csv",
    "DQN v1 Runs"
)

display(a2c_runs_df.head())
display(dqn_runs_df.head())

A2C Runs geladen: rl_navigation_project/training/run_summaries/a2c_final_training_runs.csv (3 Zeilen)
DQN v1 Runs geladen: rl_navigation_project/training/run_summaries/dqn_v1_final_evaluation_runs.csv (5 Zeilen)


,run_id,algorithm,environment,seed,timesteps,trained_steps,hyperparameters,started_at,finished_at,duration_seconds,recent_training_mean_reward,model_path,log_dir,recording_dir
0,A2C_Env1_Seed1,A2C,Env1_Simple,1,500000,500000,"{'learning_rate': 0.0003, 'n_steps': 64, 'gamm...",2026-05-08T12:32:39,2026-05-08T15:20:03,10044.63,-5.583400,rl_navigation_project/training/models/final/A2...,rl_navigation_project/training/logs/final/A2C_...,rl_navigation_project/training/recordings
1,A2C_Env1_Seed27,A2C,Env1_Simple,27,500000,500000,"{'learning_rate': 0.0003, 'n_steps': 64, 'gamm...",2026-05-08T15:45:33,2026-05-08T18:33:06,10052.49,-5.323725,rl_navigation_project/training/models/final/A2...,rl_navigation_project/training/logs/final/A2C_...,rl_navigation_project/training/recordings
2,A2C_Env1_Seed42,A2C,Env1_Simple,42,500000,500000,"{'learning_rate': 0.0003, 'n_steps': 64, 'gamm...",2026-05-08T18:45:40,2026-05-08T21:33:22,10061.81,-5.369475,rl_navigation_project/training/models/final/A2...,rl_navigation_project/training/logs/final/A2C_...,rl_navigation_project/training/recordings


,summary_file,run_id,final_analysis_label,algorithm,algorithm_version,environment,seed,timesteps,trained_steps,recent_training_mean_reward,exploration_fraction,exploration_final_eps,detected_version,model_path,log_dir,recording_dir,started_at,finished_at,duration_seconds,hyperparameters
0,DQN_Env1_Seed1_summary.json,DQN_Env1_Seed1,DQN_v1_Seed1,DQN,NaN,Env1_Simple,1,500000,500000,-5.112800,0.3,0.02,v1_original,rl_navigation_project/training/models/final/DQ...,rl_navigation_project/training/logs/final/DQN_...,rl_navigation_project/training/recordings,2026-05-08T21:43:13,2026-05-09T00:30:50,10056.50,"{'learning_rate': 0.0001, 'buffer_size': 10000..."
1,DQN_Env1_Seed27_summary.json,DQN_Env1_Seed27_v2,DQN_v1_Seed27,DQN,v2_manual_exploration,Env1_Simple,27,500000,500000,-2.170675,0.3,0.02,v1_original,rl_navigation_project/training/models/final/DQ...,rl_navigation_project/training/logs/final/DQN_...,rl_navigation_project/training/recordings,2026-05-09T16:37:21,2026-05-09T19:24:47,10045.13,"{'learning_rate': 0.0001, 'buffer_size': 10000..."
2,DQN_Env1_Seed42_summary.json,DQN_Env1_Seed42_v2,DQN_v1_Seed42,DQN,v2_manual_exploration,Env1_Simple,42,500000,500000,-5.068525,0.3,0.02,v1_original,rl_navigation_project/training/models/final/DQ...,rl_navigation_project/training/logs/final/DQN_...,rl_navigation_project/training/recordings,2026-05-09T19:36:57,2026-05-09T22:25:03,10086.20,"{'learning_rate': 0.0001, 'buffer_size': 10000..."
3,DQN_Env1_Seed72_summary.json,DQN_Env1_Seed72_v2,DQN_v1_Seed72,DQN,v2_manual_exploration,Env1_Simple,72,500000,500000,-4.466400,0.3,0.02,v1_original,rl_navigation_project/training/models/final/DQ...,rl_navigation_project/training/logs/final/DQN_...,rl_navigation_project/training/recordings,2026-05-09T22:58:46,2026-05-10T01:46:14,10047.59,"{'learning_rate': 0.0001, 'buffer_size': 10000..."
4,DQN_Env1_Seed100_summary.json,DQN_Env1_Seed100_v2,DQN_v1_Seed100,DQN,v2_manual_exploration,Env1_Simple,100,500000,500000,-3.742275,0.3,0.02,v1_original,rl_navigation_project/training/models/final/DQ...,rl_navigation_project/training/logs/final/DQN_...,rl_navigation_project/training/recordings,2026-05-10T01:59:28,2026-05-10T04:46:49,10041.53,"{'learning_rate': 0.0001, 'buffer_size': 10000..."


---

### 7. Modellpfade bereinigen und prüfen

Diese Funktionen bestimmen den Pfad zu einem gespeicherten SB3-Modell. Falls der gespeicherte Pfad in der Summary nicht direkt passt, wird zusätzlich im finalen Modellordner gesucht (sollte egtl nicht nötig sein)

In [10]:
def normalize_model_path(model_path_value) -> Path | None:
    if pd.isna(model_path_value):
        return None

    model_path = Path(str(model_path_value))

    if not model_path.is_absolute():
        model_path = PROJECT_ROOT / model_path

    return model_path


def candidate_model_paths_from_run_ids(row: pd.Series) -> list[Path]:
    candidates = []

    seed = int(row["seed"])
    algorithm = row["algorithm"]

    possible_run_ids = []

    for col in ["run_id", "original_run_id"]:
        if col in row.index and not pd.isna(row[col]):
            possible_run_ids.append(str(row[col]))

    if algorithm == "A2C":
        possible_run_ids.extend([
            f"A2C_Env1_Seed{seed}",
            f"A2C_Seed{seed}",
        ])

    if algorithm == "DQN":
        possible_run_ids.extend([
            f"DQN_Env1_Seed{seed}",
            f"DQN_Env1_Seed{seed}_v2",
            f"DQN_v1_Seed{seed}",
        ])

    possible_run_ids = list(dict.fromkeys(possible_run_ids))

    for run_id in possible_run_ids:
        candidates.extend([
            FINAL_MODEL_DIR / run_id / "final_model.zip",
            FINAL_MODEL_DIR / run_id / "final_model",
        ])

    return candidates


def find_sb3_model_path(row: pd.Series) -> Path | None:
    if "model_path" in row.index:
        candidate = normalize_model_path(row["model_path"])

        if candidate is not None:
            if candidate.exists():
                return candidate

            if candidate.with_suffix(".zip").exists():
                return candidate.with_suffix(".zip")

    for candidate in candidate_model_paths_from_run_ids(row):
        if candidate.exists():
            return candidate

    return None


def add_clean_labels(df: pd.DataFrame, algorithm: str) -> pd.DataFrame:
    if df.empty:
        return df.copy()

    out = df.copy()
    out["algorithm"] = algorithm

    if "original_run_id" not in out.columns:
        out["original_run_id"] = out["run_id"]

    if "final_analysis_label" not in out.columns:
        out["final_analysis_label"] = np.nan

    for idx, row in out.iterrows():
        seed = int(row["seed"])

        if pd.isna(row.get("final_analysis_label")):
            if algorithm == "A2C":
                out.loc[idx, "final_analysis_label"] = f"A2C_Seed{seed}"
            elif algorithm == "DQN":
                out.loc[idx, "final_analysis_label"] = f"DQN_v1_Seed{seed}"

    out["analysis_run_id"] = out["final_analysis_label"]

    out["resolved_model_path"] = out.apply(find_sb3_model_path, axis=1)
    out["model_exists"] = out["resolved_model_path"].apply(
        lambda path: path is not None and Path(path).exists()
    )

    out["resolved_model_path_display"] = out["resolved_model_path"].apply(
        lambda path: show_project_path(Path(path)) if path is not None else None
    )

    return out


a2c_models_df = add_clean_labels(a2c_runs_df, "A2C")
dqn_models_df = add_clean_labels(dqn_runs_df, "DQN")

sb3_models_df = pd.concat(
    [a2c_models_df, dqn_models_df],
    ignore_index=True,
    sort=False
)

display(sb3_models_df[[
    "analysis_run_id",
    "original_run_id",
    "algorithm",
    "seed",
    "resolved_model_path_display",
    "model_exists"
]])

,analysis_run_id,original_run_id,algorithm,seed,resolved_model_path_display,model_exists
0,A2C_Seed1,A2C_Env1_Seed1,A2C,1,rl_navigation_project/training/models/final/A2...,True
1,A2C_Seed27,A2C_Env1_Seed27,A2C,27,rl_navigation_project/training/models/final/A2...,True
2,A2C_Seed42,A2C_Env1_Seed42,A2C,42,rl_navigation_project/training/models/final/A2...,True
3,DQN_v1_Seed1,DQN_Env1_Seed1,DQN,1,rl_navigation_project/training/models/final/DQ...,True
4,DQN_v1_Seed27,DQN_Env1_Seed27_v2,DQN,27,rl_navigation_project/training/models/final/DQ...,True
5,DQN_v1_Seed42,DQN_Env1_Seed42_v2,DQN,42,rl_navigation_project/training/models/final/DQ...,True
6,DQN_v1_Seed72,DQN_Env1_Seed72_v2,DQN,72,rl_navigation_project/training/models/final/DQ...,True
7,DQN_v1_Seed100,DQN_Env1_Seed100_v2,DQN,100,rl_navigation_project/training/models/final/DQ...,True


### 7.5 Offizielle SB3-Modelle auswählen

Diese Zelle filtert auf Modelle, die wirklich existieren und evaluiert werden können. Falls ein Modellpfad fehlt, wird es angezeigt und nicht automatisch evaluiert

In [11]:
sb3_eval_models_df = sb3_models_df[sb3_models_df["model_exists"] == True].copy()
missing_models_df = sb3_models_df[sb3_models_df["model_exists"] != True].copy()

print("Gefundene SB3-Modelle:", len(sb3_eval_models_df))

display(sb3_eval_models_df[[
    "analysis_run_id",
    "algorithm",
    "seed",
    "resolved_model_path_display"
]])

if len(missing_models_df) > 0:
    print("Fehlende Modelle:")

    display(missing_models_df[[
        "analysis_run_id",
        "algorithm",
        "seed",
        "original_run_id",
        "resolved_model_path_display"
    ]])

Gefundene SB3-Modelle: 8


,analysis_run_id,algorithm,seed,resolved_model_path_display
0,A2C_Seed1,A2C,1,rl_navigation_project/training/models/final/A2...
1,A2C_Seed27,A2C,27,rl_navigation_project/training/models/final/A2...
2,A2C_Seed42,A2C,42,rl_navigation_project/training/models/final/A2...
3,DQN_v1_Seed1,DQN,1,rl_navigation_project/training/models/final/DQ...
4,DQN_v1_Seed27,DQN,27,rl_navigation_project/training/models/final/DQ...
5,DQN_v1_Seed42,DQN,42,rl_navigation_project/training/models/final/DQ...
6,DQN_v1_Seed72,DQN,72,rl_navigation_project/training/models/final/DQ...
7,DQN_v1_Seed100,DQN,100,rl_navigation_project/training/models/final/DQ...


---

### 8. Einzelne SB3-Modelle evaluieren

Bei dierser Funktion wird ein A2C- oder DQN-Modell in der aktuell geöffneten Unity-Szene evaluiert

Wichtige Änderungen:
- Run-ID enthält jetzt den Evaluationsmodus (deterministisch/stochastisch)
- Pro Run werden standardmäßig 50 Episoden evaluiert
- Action Counts werden gespeichert, um bestimmtes Verhalten wie z.b. frozen-agent- oder no-op-verhalten analysieren zu können
- beim Laden des Modells werden `custom_objects` verwendet, um Gym-/Gymnasium-/Numpy-Kompatibilitätsprobleme zu umgehen
- die info spalte wird leer gespeichert, damit keine unlesbaren Speicherobjekte in der CSV landen

In [12]:
def evaluate_single_sb3_model(
    model_row: pd.Series,
    eval_environment: str,
    n_episodes: int = EVAL_EPISODES,
    max_steps_per_episode: int = EVAL_MAX_STEPS_PER_EPISODE,
    deterministic: bool = True,
) -> pd.DataFrame:
    algorithm = model_row["algorithm"]
    seed = int(model_row["seed"])
    analysis_run_id = model_row["analysis_run_id"]
    model_path = Path(model_row["resolved_model_path"])

    if algorithm == "A2C":
        model_class = A2C
    elif algorithm == "DQN":
        model_class = DQN
    else:
        raise ValueError(f"Nicht unterstützter SB3-Algorithmus: {algorithm}")

    eval_mode_suffix = "deterministic" if deterministic else "stochastic"
    eval_run_id = f"{analysis_run_id}_eval_{eval_environment}_{eval_mode_suffix}"

    print("=" * 80)
    print("Evaluation Run:", eval_run_id)
    print("Algorithmus:", algorithm)
    print("Seed:", seed)
    print("Modell:", show_project_path(model_path))
    print("Evaluation Environment:", eval_environment)
    print("Episoden:", n_episodes)
    print("Deterministic:", deterministic)
    print("=" * 80)
    print("Bitte jetzt in Unity die passende Szene öffnen und Play drücken.")
    print("Wenn Unity verbunden ist, startet die Evaluation automatisch.")

    set_global_seeds(seed)

    env = None
    rows = []

    try:
        env = make_unity_env(seed)

        custom_objects = {
            "observation_space": env.observation_space,
            "action_space": env.action_space,
        }

        model = model_class.load(
            str(model_path),
            env=env,
            custom_objects=custom_objects,
            force_reset=True,
        )

        for episode_idx in range(1, n_episodes + 1):
            obs = reset_env(env)
            done = False

            episode_reward = 0.0
            episode_steps = 0
            action_counts = {}

            while not done and episode_steps < max_steps_per_episode:
                action, _ = model.predict(obs, deterministic=deterministic)

                action_scalar = int(np.array(action).flatten()[0])
                action_counts[action_scalar] = action_counts.get(action_scalar, 0) + 1

                obs, reward, done, info = step_env(env, action)

                episode_reward += reward_to_float(reward)
                episode_steps += 1
                done = done_to_bool(done)

            timeout_by_python_limit = episode_steps >= max_steps_per_episode and not done
            most_common_action = max(action_counts, key=action_counts.get) if action_counts else None

            rows.append({
                "eval_run_id": eval_run_id,
                "analysis_run_id": analysis_run_id,
                "original_run_id": model_row.get("original_run_id", np.nan),
                "algorithm": algorithm,
                "train_environment": TRAIN_ENV_NAME,
                "eval_environment": eval_environment,
                "seed": seed,
                "episode": episode_idx,
                "episode_reward": episode_reward,
                "episode_steps": episode_steps,
                "done": done,
                "timeout_by_python_limit": timeout_by_python_limit,
                "deterministic": deterministic,
                "eval_mode": eval_mode_suffix,
                "most_common_action": most_common_action,
                "action_counts": str(action_counts),
                "model_path": show_project_path(model_path),
                "info": "",
                "evaluated_at": pd.Timestamp.now().isoformat(timespec="seconds"),
            })

            if episode_idx % 10 == 0:
                recent_rewards = [row["episode_reward"] for row in rows[-10:]]
                recent_steps = [row["episode_steps"] for row in rows[-10:]]

                print(
                    f"{eval_run_id}: Episode {episode_idx}/{n_episodes} | "
                    f"Last10 Reward: {np.mean(recent_rewards):.3f} | "
                    f"Last10 Steps: {np.mean(recent_steps):.1f}"
                )

        eval_df = pd.DataFrame(rows)

        output_path = EVALUATION_RAW_DIR / f"{eval_run_id}.csv"
        eval_df.to_csv(output_path, index=False, encoding="utf-8")

        print("Evaluation gespeichert:")
        print(show_project_path(output_path))

        return eval_df

    finally:
        if env is not None:
            env.close()
            print(f"{eval_run_id}: Unity Environment geschlossen.")

---

### 9. Modell gezielt auswählen

Diese Funktion sucht aus der Modellübersicht ein bestimmtes Modell anhand von Algorithmus und Seed heraus und evaluiert es in der aktuell geöffneten Unity-Umgebung.

In [13]:
def get_model_row(
    algorithm: str,
    seed: int,
    models_df: pd.DataFrame = sb3_eval_models_df,
) -> pd.Series:
    matches = models_df[
        (models_df["algorithm"] == algorithm) &
        (models_df["seed"].astype(int) == int(seed))
    ]

    if len(matches) == 0:
        raise ValueError(f"Kein Modell gefunden für algorithm={algorithm}, seed={seed}")

    if len(matches) > 1:
        print("Warnung: Mehrere passende Modelle gefunden. Es wird das erste verwendet.")
        display(matches)

    return matches.iloc[0]

---

### 10. Modell gezielt evaluieren

Hier können die gezielt ausgewählten Modelle einzeln gestartet, beobachtet und aufgezeichnet werden was ziemlich wichtig für die Dokumentation des Projekts ist

In [14]:
def evaluate_selected_sb3_model(
    algorithm: str,
    seed: int,
    eval_environment: str,
    n_episodes: int = EVAL_EPISODES,
    deterministic: bool = True,
) -> pd.DataFrame:
    model_row = get_model_row(
        algorithm=algorithm,
        seed=seed,
        models_df=sb3_eval_models_df,
    )

    return evaluate_single_sb3_model(
        model_row=model_row,
        eval_environment=eval_environment,
        n_episodes=n_episodes,
        deterministic=deterministic,
    )

---

### 11. A2C Env1 - deterministisch

A2C wird zunächst deterministisch in Env1 evaluiert.  
Die deterministische Evaluation prüft, wie robust die finale Policy ist, wenn immer die bevorzugte Aktion gewählt wird.

Vor jedem Run in Unity:
- Env1 öffnen
- Behavior Type: Default
- Model: None
- EpisodeLogger Run ID passend setzen
- Max Episodes: 50

#### 11.1 A2C Seed 1 in Env1 evaluieren

Diese Zelle evaluiert das trainierte A2C Modelle deterministisch in Env1 mit Seed 1

In [32]:
eval_a2c_seed1_env1_deterministic = evaluate_selected_sb3_model(
    algorithm="A2C",
    seed=1,
    eval_environment="Env1",
    deterministic=True,
)

Evaluation Run: A2C_Seed1_eval_Env1_deterministic
Algorithmus: A2C
Seed: 1
Modell: rl_navigation_project/training/models/final/A2C_Env1_Seed1/final_model.zip
Evaluation Environment: Env1
Episoden: 50
Deterministic: True
Bitte jetzt in Unity die passende Szene öffnen und Play drücken.
Wenn Unity verbunden ist, startet die Evaluation automatisch.
Seeds gesetzt auf: 1
Seeds gesetzt auf: 1
Starte Verbindung zu Unity. Jetzt in Unity Play drücken...
[WARNING] The environment contains multiple observations. You must define allow_multiple_obs=True to receive them all. Otherwise, only the first visual observation (or vector observation ifthere are no visual observations) will be provided in the observation.
Unity Environment verbunden.
Observation Space: Box(-inf, inf, (48,), float32)
Action Space: Discrete(9)
A2C_Seed1_eval_Env1_deterministic: Episode 10/50 | Last10 Reward: -2.500 | Last10 Steps: 5000.0
A2C_Seed1_eval_Env1_deterministic: Episode 20/50 | Last10 Reward: -2.770 | Last10 Steps: 45

#### 11.2 A2C Seed 27 in Env1 evaluieren

Diese Zelle evaluiert das trainierte A2C Modelle deterministisch in Env1 mit Seed 27

In [33]:
eval_a2c_seed27_env1_deterministic = evaluate_selected_sb3_model(
    algorithm="A2C",
    seed=27,
    eval_environment="Env1",
    deterministic=True,
)

Evaluation Run: A2C_Seed27_eval_Env1_deterministic
Algorithmus: A2C
Seed: 27
Modell: rl_navigation_project/training/models/final/A2C_Env1_Seed27/final_model.zip
Evaluation Environment: Env1
Episoden: 50
Deterministic: True
Bitte jetzt in Unity die passende Szene öffnen und Play drücken.
Wenn Unity verbunden ist, startet die Evaluation automatisch.
Seeds gesetzt auf: 27
Seeds gesetzt auf: 27
Starte Verbindung zu Unity. Jetzt in Unity Play drücken...
[WARNING] The environment contains multiple observations. You must define allow_multiple_obs=True to receive them all. Otherwise, only the first visual observation (or vector observation ifthere are no visual observations) will be provided in the observation.
Unity Environment verbunden.
Observation Space: Box(-inf, inf, (48,), float32)
Action Space: Discrete(9)
A2C_Seed27_eval_Env1_deterministic: Episode 10/50 | Last10 Reward: -2.500 | Last10 Steps: 5000.0
A2C_Seed27_eval_Env1_deterministic: Episode 20/50 | Last10 Reward: -2.500 | Last10 St

#### 11.3 A2C Seed 42 in Env1 evaluieren

Diese Zelle evaluiert das trainierte A2C Modelle deterministisch in Env1 mit Seed 42.

In [34]:
eval_a2c_seed42_env1_deterministic = evaluate_selected_sb3_model(
    algorithm="A2C",
    seed=42,
    eval_environment="Env1",
    deterministic=True,
)

Evaluation Run: A2C_Seed42_eval_Env1_deterministic
Algorithmus: A2C
Seed: 42
Modell: rl_navigation_project/training/models/final/A2C_Env1_Seed42/final_model.zip
Evaluation Environment: Env1
Episoden: 50
Deterministic: True
Bitte jetzt in Unity die passende Szene öffnen und Play drücken.
Wenn Unity verbunden ist, startet die Evaluation automatisch.
Seeds gesetzt auf: 42
Seeds gesetzt auf: 42
Starte Verbindung zu Unity. Jetzt in Unity Play drücken...
[WARNING] The environment contains multiple observations. You must define allow_multiple_obs=True to receive them all. Otherwise, only the first visual observation (or vector observation ifthere are no visual observations) will be provided in the observation.
Unity Environment verbunden.
Observation Space: Box(-inf, inf, (48,), float32)
Action Space: Discrete(9)
A2C_Seed42_eval_Env1_deterministic: Episode 10/50 | Last10 Reward: -2.500 | Last10 Steps: 5000.0
A2C_Seed42_eval_Env1_deterministic: Episode 20/50 | Last10 Reward: -2.500 | Last10 St

---

### 12. A2C Env1 – stochastic

A2C wird zusätzlich stochastisch evaluiert.  
Dabei wird nicht immer die deterministisch bevorzugte Aktion genutzt, sondern aus der gelernten Policy-Verteilung gesampelt.

#### 12.1 A2C Seed 1 in Env1 evaluieren

Diese Zelle evaluiert das trainierte A2C Modelle stochastisch in Env1 mit Seed 1

In [15]:
eval_a2c_seed1_env1_stochastic = evaluate_selected_sb3_model(
    algorithm="A2C",
    seed=1,
    eval_environment="Env1",
    deterministic=False,
)

Evaluation Run: A2C_Seed1_eval_Env1_stochastic
Algorithmus: A2C
Seed: 1
Modell: rl_navigation_project/training/models/final/A2C_Env1_Seed1/final_model.zip
Evaluation Environment: Env1
Episoden: 50
Deterministic: False
Bitte jetzt in Unity die passende Szene öffnen und Play drücken.
Wenn Unity verbunden ist, startet die Evaluation automatisch.
Seeds gesetzt auf: 1
Seeds gesetzt auf: 1
Starte Verbindung zu Unity. Jetzt in Unity Play drücken...
[WARNING] The environment contains multiple observations. You must define allow_multiple_obs=True to receive them all. Otherwise, only the first visual observation (or vector observation ifthere are no visual observations) will be provided in the observation.
Unity Environment verbunden.
Observation Space: Box(-inf, inf, (48,), float32)
Action Space: Discrete(9)
A2C_Seed1_eval_Env1_stochastic: Episode 10/50 | Last10 Reward: -5.238 | Last10 Steps: 2476.9
A2C_Seed1_eval_Env1_stochastic: Episode 20/50 | Last10 Reward: -5.188 | Last10 Steps: 2375.0
A2C

#### 12.2 A2C Seed 27 in Env1 evaluieren

Diese Zelle evaluiert das trainierte A2C Modelle stochastisch in Env1 mit Seed 27

In [16]:
eval_a2c_seed27_env1_stochastic = evaluate_selected_sb3_model(
    algorithm="A2C",
    seed=27,
    eval_environment="Env1",
    deterministic=False,
)

Evaluation Run: A2C_Seed27_eval_Env1_stochastic
Algorithmus: A2C
Seed: 27
Modell: rl_navigation_project/training/models/final/A2C_Env1_Seed27/final_model.zip
Evaluation Environment: Env1
Episoden: 50
Deterministic: False
Bitte jetzt in Unity die passende Szene öffnen und Play drücken.
Wenn Unity verbunden ist, startet die Evaluation automatisch.
Seeds gesetzt auf: 27
Seeds gesetzt auf: 27
Starte Verbindung zu Unity. Jetzt in Unity Play drücken...
[WARNING] The environment contains multiple observations. You must define allow_multiple_obs=True to receive them all. Otherwise, only the first visual observation (or vector observation ifthere are no visual observations) will be provided in the observation.
Unity Environment verbunden.
Observation Space: Box(-inf, inf, (48,), float32)
Action Space: Discrete(9)
A2C_Seed27_eval_Env1_stochastic: Episode 10/50 | Last10 Reward: -5.508 | Last10 Steps: 2016.1
A2C_Seed27_eval_Env1_stochastic: Episode 20/50 | Last10 Reward: -5.726 | Last10 Steps: 245

#### 12.3 A2C Seed 42 in Env1 evaluieren

Diese Zelle evaluiert das trainierte A2C Modelle stochastisch in Env1 mit Seed 42.

In [14]:
eval_a2c_seed42_env1_stochastic = evaluate_selected_sb3_model(
    algorithm="A2C",
    seed=42,
    eval_environment="Env1",
    deterministic=False,
)

Evaluation Run: A2C_Seed42_eval_Env1_stochastic
Algorithmus: A2C
Seed: 42
Modell: rl_navigation_project/training/models/final/A2C_Env1_Seed42/final_model.zip
Evaluation Environment: Env1
Episoden: 50
Deterministic: False
Bitte jetzt in Unity die passende Szene öffnen und Play drücken.
Wenn Unity verbunden ist, startet die Evaluation automatisch.
Seeds gesetzt auf: 42
Seeds gesetzt auf: 42
Starte Verbindung zu Unity. Jetzt in Unity Play drücken...
[WARNING] The environment contains multiple observations. You must define allow_multiple_obs=True to receive them all. Otherwise, only the first visual observation (or vector observation ifthere are no visual observations) will be provided in the observation.
Unity Environment verbunden.
Observation Space: Box(-inf, inf, (48,), float32)
Action Space: Discrete(9)
A2C_Seed42_eval_Env1_stochastic: Episode 10/50 | Last10 Reward: -5.934 | Last10 Steps: 1868.4
A2C_Seed42_eval_Env1_stochastic: Episode 20/50 | Last10 Reward: -4.513 | Last10 Steps: 302

---

### 13. DQN Env1 - deterministisch

DQN wird zunächst deterministisch in Env1 evaluiert.  
Diese Evaluation dient vor allem als Robustheitsdiagnose der greedy Policy. Das ist besonders relevant, da bei DQN Seed 27 bereits in ersten Testläufen ein Frozen-Agent-Verhalten in Zielnähe beobachtet wurde.

*Hinweis: Das beobachtete Frozen-Agent-Verhalten ist der zentrale Grund dafür, zusätzlich zur deterministischen Evaluation auch eine stochastische Evaluation durchzuführen. In der deterministischen Evaluation wählt der Agent stets die greedy Action. Wenn diese Aktion in Zielnähe zu einem No-Op oder einer ineffektiven Bewegung führt, kann der Agent reproduzierbar in einer suboptimalen Situation einfrieren. Erste Testläufe zeigten genau dieses Muster.Während deterministische Durchläufe dazu führten, dass der Agent in direkter Zielnähe stehen blieb, konnten in stochastischen Durchläufen Goal-Events erreicht werden. Deshalb werden DQN-Modelle in beiden Modi evaluiert.*

*Hinweis 2: Aus Gründen der wissenschaftlichen Nachvollziehbarkeit und Konsistenz wird auch A2C in beiden Modi (deterministisch und stochastisch) evaluiert. Diese Entscheidung wurde erst nach der Beobachtung und Analyse der DQN-Ergebnisse getroffen, auch wenn A2C in der Notebook-Struktur vor DQN ausgewertet wird.*

#### 13.1 DQN Seed 1 in Env1 evaluieren

Diese Zelle evaluiert das trainierte DQN Modelle deterministisch in Env1 mit Seed 1

In [15]:
eval_dqn_seed1_env1_deterministic = evaluate_selected_sb3_model(
    algorithm="DQN",
    seed=1,
    eval_environment="Env1",
    deterministic=True,
)

Evaluation Run: DQN_v1_Seed1_eval_Env1_deterministic
Algorithmus: DQN
Seed: 1
Modell: rl_navigation_project/training/models/final/DQN_Env1_Seed1/final_model.zip
Evaluation Environment: Env1
Episoden: 50
Deterministic: True
Bitte jetzt in Unity die passende Szene öffnen und Play drücken.
Wenn Unity verbunden ist, startet die Evaluation automatisch.
Seeds gesetzt auf: 1
Seeds gesetzt auf: 1
Starte Verbindung zu Unity. Jetzt in Unity Play drücken...
[WARNING] The environment contains multiple observations. You must define allow_multiple_obs=True to receive them all. Otherwise, only the first visual observation (or vector observation ifthere are no visual observations) will be provided in the observation.
Unity Environment verbunden.
Observation Space: Box(-inf, inf, (48,), float32)
Action Space: Discrete(9)
DQN_v1_Seed1_eval_Env1_deterministic: Episode 10/50 | Last10 Reward: -2.751 | Last10 Steps: 4502.0
DQN_v1_Seed1_eval_Env1_deterministic: Episode 20/50 | Last10 Reward: -3.574 | Last10 

#### 13.2 DQN Seed 27 in Env1 evaluieren

Diese Zelle evaluiert das trainierte DQN Modelle deterministisch in Env1 mit Seed 27

In [16]:
eval_dqn_seed27_env1_deterministic = evaluate_selected_sb3_model(
    algorithm="DQN",
    seed=27,
    eval_environment="Env1",
    deterministic=True,
)

Evaluation Run: DQN_v1_Seed27_eval_Env1_deterministic
Algorithmus: DQN
Seed: 27
Modell: rl_navigation_project/training/models/final/DQN_Env1_Seed27/final_model.zip
Evaluation Environment: Env1
Episoden: 50
Deterministic: True
Bitte jetzt in Unity die passende Szene öffnen und Play drücken.
Wenn Unity verbunden ist, startet die Evaluation automatisch.
Seeds gesetzt auf: 27
Seeds gesetzt auf: 27
Starte Verbindung zu Unity. Jetzt in Unity Play drücken...
[WARNING] The environment contains multiple observations. You must define allow_multiple_obs=True to receive them all. Otherwise, only the first visual observation (or vector observation ifthere are no visual observations) will be provided in the observation.
Unity Environment verbunden.
Observation Space: Box(-inf, inf, (48,), float32)
Action Space: Discrete(9)
DQN_v1_Seed27_eval_Env1_deterministic: Episode 10/50 | Last10 Reward: -3.758 | Last10 Steps: 2515.1
DQN_v1_Seed27_eval_Env1_deterministic: Episode 20/50 | Last10 Reward: -3.758 | 

#### 13.3 DQN Seed 42 in Env1 evaluieren

Diese Zelle evaluiert das trainierte DQN Modelle deterministisch in Env1 mit Seed 42

In [17]:
eval_dqn_seed42_env1_deterministic = evaluate_selected_sb3_model(
    algorithm="DQN",
    seed=42,
    eval_environment="Env1",
    deterministic=True,
)

Evaluation Run: DQN_v1_Seed42_eval_Env1_deterministic
Algorithmus: DQN
Seed: 42
Modell: rl_navigation_project/training/models/final/DQN_Env1_Seed42/final_model.zip
Evaluation Environment: Env1
Episoden: 50
Deterministic: True
Bitte jetzt in Unity die passende Szene öffnen und Play drücken.
Wenn Unity verbunden ist, startet die Evaluation automatisch.
Seeds gesetzt auf: 42
Seeds gesetzt auf: 42
Starte Verbindung zu Unity. Jetzt in Unity Play drücken...
[WARNING] The environment contains multiple observations. You must define allow_multiple_obs=True to receive them all. Otherwise, only the first visual observation (or vector observation ifthere are no visual observations) will be provided in the observation.
Unity Environment verbunden.
Observation Space: Box(-inf, inf, (48,), float32)
Action Space: Discrete(9)
DQN_v1_Seed42_eval_Env1_deterministic: Episode 10/50 | Last10 Reward: -3.565 | Last10 Steps: 3129.0
DQN_v1_Seed42_eval_Env1_deterministic: Episode 20/50 | Last10 Reward: -4.670 | 

#### 13.4 DQN Seed 72 in Env1 evaluieren

Diese Zelle evaluiert das trainierte DQN Modelle deterministisch in Env1 mit Seed 72

In [18]:
eval_dqn_seed72_env1_deterministic = evaluate_selected_sb3_model(
    algorithm="DQN",
    seed=72,
    eval_environment="Env1",
    deterministic=True,
)

Evaluation Run: DQN_v1_Seed72_eval_Env1_deterministic
Algorithmus: DQN
Seed: 72
Modell: rl_navigation_project/training/models/final/DQN_Env1_Seed72/final_model.zip
Evaluation Environment: Env1
Episoden: 50
Deterministic: True
Bitte jetzt in Unity die passende Szene öffnen und Play drücken.
Wenn Unity verbunden ist, startet die Evaluation automatisch.
Seeds gesetzt auf: 72
Seeds gesetzt auf: 72
Starte Verbindung zu Unity. Jetzt in Unity Play drücken...
[WARNING] The environment contains multiple observations. You must define allow_multiple_obs=True to receive them all. Otherwise, only the first visual observation (or vector observation ifthere are no visual observations) will be provided in the observation.
Unity Environment verbunden.
Observation Space: Box(-inf, inf, (48,), float32)
Action Space: Discrete(9)
DQN_v1_Seed72_eval_Env1_deterministic: Episode 10/50 | Last10 Reward: -4.596 | Last10 Steps: 1192.7
DQN_v1_Seed72_eval_Env1_deterministic: Episode 20/50 | Last10 Reward: -3.803 | 

#### 13.5 DQN Seed 100 in Env1 evaluieren

Diese Zelle evaluiert das trainierte DQN Modelle deterministisch in Env1 mit Seed 100

In [19]:
eval_dqn_seed100_env1_deterministic = evaluate_selected_sb3_model(
    algorithm="DQN",
    seed=100,
    eval_environment="Env1",
    deterministic=True,
)

Evaluation Run: DQN_v1_Seed100_eval_Env1_deterministic
Algorithmus: DQN
Seed: 100
Modell: rl_navigation_project/training/models/final/DQN_Env1_Seed100/final_model.zip
Evaluation Environment: Env1
Episoden: 50
Deterministic: True
Bitte jetzt in Unity die passende Szene öffnen und Play drücken.
Wenn Unity verbunden ist, startet die Evaluation automatisch.
Seeds gesetzt auf: 100
Seeds gesetzt auf: 100
Starte Verbindung zu Unity. Jetzt in Unity Play drücken...
[WARNING] The environment contains multiple observations. You must define allow_multiple_obs=True to receive them all. Otherwise, only the first visual observation (or vector observation ifthere are no visual observations) will be provided in the observation.
Unity Environment verbunden.
Observation Space: Box(-inf, inf, (48,), float32)
Action Space: Discrete(9)
DQN_v1_Seed100_eval_Env1_deterministic: Episode 10/50 | Last10 Reward: -3.275 | Last10 Steps: 3549.7
DQN_v1_Seed100_eval_Env1_deterministic: Episode 20/50 | Last10 Reward: -3

---

### 14. DQN Env1 – stochastic

Wie im Abschnitt 13. schon erklärt wird DQN zusätzlich stochastisch evaluiert.  
Diese Evaluation prüft, ob die Modelle grundsätzlich zielrelevantes Verhalten gelernt haben, auch wenn die rein greedy Policy nicht robust ist.

Die Ergebnisse werden getrennt von der deterministischen Evaluation berichtet.

#### 14.1 DQN Seed 1 in Env1 evaluieren

Diese Zelle evaluiert das trainierte DQN Modelle stocahstisch in Env1 mit Seed 1

In [20]:
eval_dqn_seed1_env1_stochastic = evaluate_selected_sb3_model(
    algorithm="DQN",
    seed=1,
    eval_environment="Env1",
    deterministic=False,
)

Evaluation Run: DQN_v1_Seed1_eval_Env1_stochastic
Algorithmus: DQN
Seed: 1
Modell: rl_navigation_project/training/models/final/DQN_Env1_Seed1/final_model.zip
Evaluation Environment: Env1
Episoden: 50
Deterministic: False
Bitte jetzt in Unity die passende Szene öffnen und Play drücken.
Wenn Unity verbunden ist, startet die Evaluation automatisch.
Seeds gesetzt auf: 1
Seeds gesetzt auf: 1
Starte Verbindung zu Unity. Jetzt in Unity Play drücken...
[WARNING] The environment contains multiple observations. You must define allow_multiple_obs=True to receive them all. Otherwise, only the first visual observation (or vector observation ifthere are no visual observations) will be provided in the observation.
Unity Environment verbunden.
Observation Space: Box(-inf, inf, (48,), float32)
Action Space: Discrete(9)
DQN_v1_Seed1_eval_Env1_stochastic: Episode 10/50 | Last10 Reward: -4.699 | Last10 Steps: 2397.0
DQN_v1_Seed1_eval_Env1_stochastic: Episode 20/50 | Last10 Reward: -3.511 | Last10 Steps: 3

#### 14.2 DQN Seed 27 in Env1 evaluieren

Diese Zelle evaluiert das trainierte DQN Modelle stocahstisch in Env1 mit Seed 27

In [21]:
eval_dqn_seed27_env1_stochastic = evaluate_selected_sb3_model(
    algorithm="DQN",
    seed=27,
    eval_environment="Env1",
    deterministic=False,
)

Evaluation Run: DQN_v1_Seed27_eval_Env1_stochastic
Algorithmus: DQN
Seed: 27
Modell: rl_navigation_project/training/models/final/DQN_Env1_Seed27/final_model.zip
Evaluation Environment: Env1
Episoden: 50
Deterministic: False
Bitte jetzt in Unity die passende Szene öffnen und Play drücken.
Wenn Unity verbunden ist, startet die Evaluation automatisch.
Seeds gesetzt auf: 27
Seeds gesetzt auf: 27
Starte Verbindung zu Unity. Jetzt in Unity Play drücken...
[WARNING] The environment contains multiple observations. You must define allow_multiple_obs=True to receive them all. Otherwise, only the first visual observation (or vector observation ifthere are no visual observations) will be provided in the observation.
Unity Environment verbunden.
Observation Space: Box(-inf, inf, (48,), float32)
Action Space: Discrete(9)
DQN_v1_Seed27_eval_Env1_stochastic: Episode 10/50 | Last10 Reward: -4.841 | Last10 Steps: 1682.0
DQN_v1_Seed27_eval_Env1_stochastic: Episode 20/50 | Last10 Reward: -1.853 | Last10 S

#### 14.3 DQN Seed 42 in Env1 evaluieren

Diese Zelle evaluiert das trainierte DQN Modelle stocahstisch in Env1 mit Seed 42

In [22]:
eval_dqn_seed42_env1_stochastic = evaluate_selected_sb3_model(
    algorithm="DQN",
    seed=42,
    eval_environment="Env1",
    deterministic=False,
)

Evaluation Run: DQN_v1_Seed42_eval_Env1_stochastic
Algorithmus: DQN
Seed: 42
Modell: rl_navigation_project/training/models/final/DQN_Env1_Seed42/final_model.zip
Evaluation Environment: Env1
Episoden: 50
Deterministic: False
Bitte jetzt in Unity die passende Szene öffnen und Play drücken.
Wenn Unity verbunden ist, startet die Evaluation automatisch.
Seeds gesetzt auf: 42
Seeds gesetzt auf: 42
Starte Verbindung zu Unity. Jetzt in Unity Play drücken...
[WARNING] The environment contains multiple observations. You must define allow_multiple_obs=True to receive them all. Otherwise, only the first visual observation (or vector observation ifthere are no visual observations) will be provided in the observation.
Unity Environment verbunden.
Observation Space: Box(-inf, inf, (48,), float32)
Action Space: Discrete(9)
DQN_v1_Seed42_eval_Env1_stochastic: Episode 10/50 | Last10 Reward: -4.975 | Last10 Steps: 1949.9
DQN_v1_Seed42_eval_Env1_stochastic: Episode 20/50 | Last10 Reward: -4.604 | Last10 S

#### 14.4 DQN Seed 72 in Env1 evaluieren

Diese Zelle evaluiert das trainierte DQN Modelle stocahstisch in Env1 mit Seed 72

In [23]:
eval_dqn_seed72_env1_stochastic = evaluate_selected_sb3_model(
    algorithm="DQN",
    seed=72,
    eval_environment="Env1",
    deterministic=False,
)

Evaluation Run: DQN_v1_Seed72_eval_Env1_stochastic
Algorithmus: DQN
Seed: 72
Modell: rl_navigation_project/training/models/final/DQN_Env1_Seed72/final_model.zip
Evaluation Environment: Env1
Episoden: 50
Deterministic: False
Bitte jetzt in Unity die passende Szene öffnen und Play drücken.
Wenn Unity verbunden ist, startet die Evaluation automatisch.
Seeds gesetzt auf: 72
Seeds gesetzt auf: 72
Starte Verbindung zu Unity. Jetzt in Unity Play drücken...
[WARNING] The environment contains multiple observations. You must define allow_multiple_obs=True to receive them all. Otherwise, only the first visual observation (or vector observation ifthere are no visual observations) will be provided in the observation.
Unity Environment verbunden.
Observation Space: Box(-inf, inf, (48,), float32)
Action Space: Discrete(9)
DQN_v1_Seed72_eval_Env1_stochastic: Episode 10/50 | Last10 Reward: -4.603 | Last10 Steps: 1206.4
DQN_v1_Seed72_eval_Env1_stochastic: Episode 20/50 | Last10 Reward: -4.040 | Last10 S

#### 14.5 DQN Seed 100 in Env1 evaluieren

Diese Zelle evaluiert das trainierte DQN Modelle stocahstisch in Env1 mit Seed 100

In [24]:
eval_dqn_seed100_env1_stochastic = evaluate_selected_sb3_model(
    algorithm="DQN",
    seed=100,
    eval_environment="Env1",
    deterministic=False,
)

Evaluation Run: DQN_v1_Seed100_eval_Env1_stochastic
Algorithmus: DQN
Seed: 100
Modell: rl_navigation_project/training/models/final/DQN_Env1_Seed100/final_model.zip
Evaluation Environment: Env1
Episoden: 50
Deterministic: False
Bitte jetzt in Unity die passende Szene öffnen und Play drücken.
Wenn Unity verbunden ist, startet die Evaluation automatisch.
Seeds gesetzt auf: 100
Seeds gesetzt auf: 100
Starte Verbindung zu Unity. Jetzt in Unity Play drücken...
[WARNING] The environment contains multiple observations. You must define allow_multiple_obs=True to receive them all. Otherwise, only the first visual observation (or vector observation ifthere are no visual observations) will be provided in the observation.
Unity Environment verbunden.
Observation Space: Box(-inf, inf, (48,), float32)
Action Space: Discrete(9)
DQN_v1_Seed100_eval_Env1_stochastic: Episode 10/50 | Last10 Reward: -3.548 | Last10 Steps: 3096.7
DQN_v1_Seed100_eval_Env1_stochastic: Episode 20/50 | Last10 Reward: -3.811 | L

---

### 15. PPO Planning 

PPO wird separat evaluiert, da PPO über Unity ML-Agents trainiert wurde und nicht als Stable-Baselines3-Modell geladen wird.

Für PPO gilt:
- ONNX-Modell in Unity beim MazeAgent eintragen
- Behavior Type: Inference Only
- passende Szene öffnen
- EpisodeLogger aktivieren
- Max Episodes: 50

PPO wird nicht in dieselbe deterministic/stochastic-Struktur wie A2C und DQN gepresst.  
Die Ergebnisse werden separat als native Unity-ML-Agents-Inference-Evaluation dokumentiert.

In [15]:
ppo_eval_plan_df = pd.DataFrame([
    {"algorithm": "PPO", "seed": 1, "eval_environment": "Env1", "episodes": EVAL_EPISODES, "run_id": "PPO_Seed1_eval_Env1"},
    {"algorithm": "PPO", "seed": 27, "eval_environment": "Env1", "episodes": EVAL_EPISODES, "run_id": "PPO_Seed27_eval_Env1"},
    {"algorithm": "PPO", "seed": 42, "eval_environment": "Env1", "episodes": EVAL_EPISODES, "run_id": "PPO_Seed42_eval_Env1"},
    {"algorithm": "PPO", "seed": 72, "eval_environment": "Env1", "episodes": EVAL_EPISODES, "run_id": "PPO_Seed72_eval_Env1"},
    {"algorithm": "PPO", "seed": 100, "eval_environment": "Env1", "episodes": EVAL_EPISODES, "run_id": "PPO_Seed100_eval_Env1"},
])

display(ppo_eval_plan_df)

,algorithm,seed,eval_environment,episodes,run_id
0,PPO,1,Env1,50,PPO_Seed1_eval_Env1
1,PPO,27,Env1,50,PPO_Seed27_eval_Env1
2,PPO,42,Env1,50,PPO_Seed42_eval_Env1
3,PPO,72,Env1,50,PPO_Seed72_eval_Env1
4,PPO,100,Env1,50,PPO_Seed100_eval_Env1


---

## Zusatz (Z1): PPO ONNX-Export für Unity Inference

PPO wurde über Unity ML-Agents trainiert. Nach dem Training lagen zunächst nur PyTorch-Checkpoints (`.pt`) in den jeweiligen Trainingsordnern vor. Diese Dateien können von Unity jedoch nicht direkt im Feld `Behavior Parameters > Model` ausgewählt werden 

Für die finale PPO-Evaluation über Unity Inference wurden die trainierten PPO-Modelle daher nachträglich als ONNX-Dateien exportiert. Dafür wurde jeder PPO-Run mit `mlagents-learn --resume` kurz erneut geladen. Nach der Verbindung mit Unity exportierte ML-Agents automatisch ein `.onnx`-Modell in den jeweiligen Ergebnisordner.

Wichtig:
- der EpisodeLogger war während des ONNX-Exports deaktiviert
- es wurde keine Aufnahme gestartet
- der MazeAgent stand auf `Behavior Type: Default`
- es wurde kein ONNX-Modell im Agenten gesetzt
- der Export wurde nur genutzt, um die finalen ONNX-Dateien für die spätere Inference-Evaluation zu erzeugen

Die exportierten ONNX-Dateien wurden anschließend in den Unity-Assets-Ordner kopiert:

`MazeAgent/Assets/Models`

Dadurch können die Modelle in Unity im Feld `Behavior Parameters > Model` ausgewählt werden.

### Zusatz (Z2): Bash Commands für den nachträglichen ONNX-Export

Die folgenden Befehle wurden im aktivierten `rl_project` Environment im Projekt-Root ausgeführt

Zuerst wurden die nötigen Paketversionen korrigiert, da ONNX und ML-Agents unterschiedliche Anforderungen an `protobuf` und `numpy` hatten.

```bash
python -m pip install --force-reinstall onnx==1.13.1 protobuf==3.20.3
python -m pip install --force-reinstall numpy==1.23.5
```

Danach wurden die Versionen geprüft mit dem folgenden Zielzustand:
- numpy 1.23.5
- onnx 1.13.1   
- protobuf 3.20.3

```bash
python -c "import numpy as np; print('numpy', np.__version__)"
python -c "import onnx; print('onnx', onnx.__version__)"
python -c "import google.protobuf; print('protobuf', google.protobuf.__version__)"
```

#### PPO Seed 1 – ONNX-Export

Für Seed 1 wurde der bestehende ML-Agents-Run kurz mit `--resume` geladen. Nach dem Drücken von Play in Unity wurde automatisch ein ONNX-Modell exportiert.

```bash
mlagents-learn training/configs/ppo_env1_config.yaml --run-id=PPO_Env1_Seed1 --resume --results-dir=training/logs/final
```

Im Terminal wurde anschließend bestätigt, dass das Modell exportiert wurde:

`training/logs/final/PPO_Env1_Seed1/MazeAgent.onnx`

Diese Datei wurde danach in den Unity-Assets-Ordner kopiert:

```bash
cp training/logs/final/PPO_Env1_Seed1/MazeAgent.onnx MazeAgent/Assets/Models/PPO_Env1_Seed1.onnx
```


### PPO Seed 27 – ONNX-Export

```bash
mlagents-learn training/configs/ppo_env1_config.yaml --run-id=PPO_Env1_Seed27 --resume --results-dir=training/logs/final
```


Danach erst:

```bash
cp training/logs/final/PPO_Env1_Seed27/MazeAgent.onnx MazeAgent/Assets/Models/PPO_Env1_Seed27.onnx
```

### PPO Seed 42 – ONNX-Export

```bash
mlagents-learn training/configs/ppo_env1_config.yaml --run-id=PPO_Env1_Seed42 --resume --results-dir=training/logs/final
```

Danach erst:

```bash
cp training/logs/final/PPO_Env1_Seed42/MazeAgent.onnx MazeAgent/Assets/Models/PPO_Env1_Seed42.onnx
```

### PPO Seed 72 – ONNX-Export

```bash
mlagents-learn training/configs/ppo_env1_config.yaml --run-id=PPO_Env1_Seed72 --resume --results-dir=training/logs/final
```

Danach erst:

```bash
cp training/logs/final/PPO_Env1_Seed72/MazeAgent.onnx MazeAgent/Assets/Models/PPO_Env1_Seed72.onnx
```

### PPO Seed 100 – ONNX-Export

```bash
mlagents-learn training/configs/ppo_env1_config.yaml --run-id=PPO_Env1_Seed100 --resume --results-dir=training/logs/final
```

Danach erst:

```bash
cp training/logs/final/PPO_Env1_Seed100/MazeAgent.onnx MazeAgent/Assets/Models/PPO_Env1_Seed100.onnx
```

**Erwarte Dateien:**

MazeAgent/Assets/Models/PPO_Env1_Seed1.onnx

MazeAgent/Assets/Models/PPO_Env1_Seed100.onnx

MazeAgent/Assets/Models/PPO_Env1_Seed27.onnx

MazeAgent/Assets/Models/PPO_Env1_Seed42.onnx

MazeAgent/Assets/Models/PPO_Env1_Seed72.onnx

---

### 16. PPO Evaluation über Unity ML-Agents Inference

Nach dem ONNX-Export werden die PPO-Modelle direkt in Unity evaluiert.

Für jeden Seed wird in Unity eingestellt:

- Szene: `Env1`
- MazeAgent → Behavior Parameters:
  - Behavior Type: `Inference Only`
  - Model: passendes ONNX-Modell, z. B. `PPO_Env1_Seed1`
- EpisodeLogger:
  - Enable Logging: aktiv
  - Use Max Episodes: aktiv
  - Max Episodes: 50
  - Stop Play Mode: aktiv
  - Run Id: passend zum Seed, z. B. `PPO_Seed1_eval_Env1`
  - Algorithm Name: `PPO`
  - Environment Name: `Env1`

Die PPO-Evaluation läuft damit vollständig in Unity. Die erzeugten CSV-Dateien werden zunächst unter folgendem Ordner gespeichert:

`MazeAgent/training/evaluation_logs`

Für die finale Auswertung werden diese CSV-Dateien anschließend in den finalen Evaluation-Ordner kopiert:

`training/evaluation_results/raw`

In [16]:
# PPO Unity Logger schreibt die CSV-Dateien zunächst hierhin
PPO_UNITY_LOG_DIR = PROJECT_ROOT / "MazeAgent" / "training" / "evaluation_logs"

def copy_ppo_eval_file(run_id: str) -> pd.DataFrame:
    source_path = PPO_UNITY_LOG_DIR / f"{run_id}.csv"
    target_path = EVALUATION_RAW_DIR / f"{run_id}.csv"

    if not source_path.exists():
        raise FileNotFoundError(
            f"PPO CSV nicht gefunden: {show_project_path(source_path)}"
        )

    raw_df = pd.read_csv(source_path)
    normalized_df = raw_df.copy()

    normalized_df = normalized_df.rename(columns={
        "run_id": "eval_run_id",
        "environment": "eval_environment",
        "total_reward": "episode_reward",
        "steps": "episode_steps",
    })

    normalized_df["eval_environment"] = normalized_df["eval_environment"].replace({
        "Env1_Simple": "Env1",
        "Env2_Simple": "Env2",
        "Env3_Simple": "Env3",
    })

    seed = int(normalized_df["seed"].iloc[0])

    normalized_df["analysis_run_id"] = f"PPO_Seed{seed}"
    normalized_df["original_run_id"] = f"PPO_Env1_Seed{seed}"
    normalized_df["algorithm"] = "PPO"
    normalized_df["train_environment"] = TRAIN_ENV_NAME
    normalized_df["deterministic"] = np.nan
    normalized_df["eval_mode"] = "ml_agents_inference"
    normalized_df["timeout_by_python_limit"] = False
    normalized_df["most_common_action"] = np.nan
    normalized_df["action_counts"] = ""
    normalized_df["model_path"] = f"MazeAgent/Assets/Models/PPO_Env1_Seed{seed}.onnx"
    normalized_df["info"] = ""
    normalized_df["evaluated_at"] = pd.Timestamp.now().isoformat(timespec="seconds")
    normalized_df["done"] = True

    # <<< WICHTIG: PPO-Ergebnisflags sauber übernehmen
    normalized_df["success_flag"] = normalized_df["success"].astype(int)
    normalized_df["wall_hit_flag"] = normalized_df["wall_hit"].astype(int)
    normalized_df["timeout_flag"] = normalized_df["timeout"].astype(int)

    final_cols = [
        "eval_run_id",
        "analysis_run_id",
        "original_run_id",
        "algorithm",
        "train_environment",
        "eval_environment",
        "seed",
        "episode",
        "episode_reward",
        "episode_steps",
        "done",
        "timeout_by_python_limit",
        "deterministic",
        "eval_mode",
        "most_common_action",
        "action_counts",
        "model_path",
        "info",
        "evaluated_at",
        "success_flag",
        "wall_hit_flag",
        "timeout_flag",
    ]

    normalized_df = normalized_df[final_cols]
    normalized_df.to_csv(target_path, index=False, encoding="utf-8")

    print("PPO CSV normalisiert und kopiert:")
    print("Von:", show_project_path(source_path))
    print("Nach:", show_project_path(target_path))
    print("Geladene Episoden:", len(normalized_df))

    display(normalized_df.head())

    return normalized_df

#### 16.1 PPO Env1 – Seed 1

Unity-Einstellungen:

- ONNX-Modell: `PPO_Env1_Seed1.onnx`
- Behavior Type: `Inference Only`
- EpisodeLogger Run Id: `PPO_Seed1_eval_Env1`
- Algorithm Name: `PPO`
- Environment Name: `Env1`
- Max Episodes: `50`

Nach Abschluss des Unity-Runs wird die CSV-Datei aus `MazeAgent/training/evaluation_logs` in den finalen Evaluation-Ordner kopiert.

```bash
#Erwartete Datei nach dem Unity-Run:
MazeAgent/training/evaluation_logs/PPO_Seed1_eval_Env1.csv

# Zielordner für die finale Auswertung:
training/evaluation_results/raw/PPO_Seed1_eval_Env1.csv
```

In [41]:
ppo_seed1_env1_df = copy_ppo_eval_file("PPO_Seed1_eval_Env1")

PPO CSV normalisiert und kopiert:
Von: rl_navigation_project/MazeAgent/training/evaluation_logs/PPO_Seed1_eval_Env1.csv
Nach: rl_navigation_project/training/evaluation_results/raw/PPO_Seed1_eval_Env1.csv
Geladene Episoden: 50


,eval_run_id,analysis_run_id,original_run_id,algorithm,train_environment,eval_environment,seed,episode,episode_reward,episode_steps,...,deterministic,eval_mode,most_common_action,action_counts,model_path,info,evaluated_at,success_flag,wall_hit_flag,timeout_flag
0,PPO_Seed1_eval_Env1,PPO_Seed1,PPO_Env1_Seed1,PPO,Env1,Env1,1,1,-2.4994,4999,...,NaN,ml_agents_inference,NaN,,MazeAgent/Assets/Models/PPO_Env1_Seed1.onnx,,2026-05-14T18:50:05,0,0,1
1,PPO_Seed1_eval_Env1,PPO_Seed1,PPO_Env1_Seed1,PPO,Env1,Env1,1,2,-2.4994,4999,...,NaN,ml_agents_inference,NaN,,MazeAgent/Assets/Models/PPO_Env1_Seed1.onnx,,2026-05-14T18:50:05,0,0,1
2,PPO_Seed1_eval_Env1,PPO_Seed1,PPO_Env1_Seed1,PPO,Env1,Env1,1,3,-2.4994,4999,...,NaN,ml_agents_inference,NaN,,MazeAgent/Assets/Models/PPO_Env1_Seed1.onnx,,2026-05-14T18:50:05,0,0,1
3,PPO_Seed1_eval_Env1,PPO_Seed1,PPO_Env1_Seed1,PPO,Env1,Env1,1,4,-2.4994,4999,...,NaN,ml_agents_inference,NaN,,MazeAgent/Assets/Models/PPO_Env1_Seed1.onnx,,2026-05-14T18:50:05,0,0,1
4,PPO_Seed1_eval_Env1,PPO_Seed1,PPO_Env1_Seed1,PPO,Env1,Env1,1,5,-2.4994,4999,...,NaN,ml_agents_inference,NaN,,MazeAgent/Assets/Models/PPO_Env1_Seed1.onnx,,2026-05-14T18:50:05,0,0,1


#### 16.2 PPO Env1 – Seed 27

Unity-Einstellungen:

- ONNX-Modell: `PPO_Env1_Seed27.onnx`
- Behavior Type: `Inference Only`
- EpisodeLogger Run Id: `PPO_Seed27_eval_Env1`
- Algorithm Name: `PPO`
- Environment Name: `Env1`
- Max Episodes: `50`

```bash
# Erwartete Datei nach dem Unity-Run:
MazeAgent/training/evaluation_logs/PPO_Seed27_eval_Env1.csv

# Zielordner für die finale Auswertung:
training/evaluation_results/raw/PPO_Seed27_eval_Env1.csv
```

In [42]:
ppo_seed27_env1_df = copy_ppo_eval_file("PPO_Seed27_eval_Env1")

PPO CSV normalisiert und kopiert:
Von: rl_navigation_project/MazeAgent/training/evaluation_logs/PPO_Seed27_eval_Env1.csv
Nach: rl_navigation_project/training/evaluation_results/raw/PPO_Seed27_eval_Env1.csv
Geladene Episoden: 50


,eval_run_id,analysis_run_id,original_run_id,algorithm,train_environment,eval_environment,seed,episode,episode_reward,episode_steps,...,deterministic,eval_mode,most_common_action,action_counts,model_path,info,evaluated_at,success_flag,wall_hit_flag,timeout_flag
0,PPO_Seed27_eval_Env1,PPO_Seed27,PPO_Env1_Seed27,PPO,Env1,Env1,27,1,-2.4994,4999,...,NaN,ml_agents_inference,NaN,,MazeAgent/Assets/Models/PPO_Env1_Seed27.onnx,,2026-05-14T18:50:19,0,0,1
1,PPO_Seed27_eval_Env1,PPO_Seed27,PPO_Env1_Seed27,PPO,Env1,Env1,27,2,-2.4994,4999,...,NaN,ml_agents_inference,NaN,,MazeAgent/Assets/Models/PPO_Env1_Seed27.onnx,,2026-05-14T18:50:19,0,0,1
2,PPO_Seed27_eval_Env1,PPO_Seed27,PPO_Env1_Seed27,PPO,Env1,Env1,27,3,-2.4994,4999,...,NaN,ml_agents_inference,NaN,,MazeAgent/Assets/Models/PPO_Env1_Seed27.onnx,,2026-05-14T18:50:19,0,0,1
3,PPO_Seed27_eval_Env1,PPO_Seed27,PPO_Env1_Seed27,PPO,Env1,Env1,27,4,-2.4994,4999,...,NaN,ml_agents_inference,NaN,,MazeAgent/Assets/Models/PPO_Env1_Seed27.onnx,,2026-05-14T18:50:19,0,0,1
4,PPO_Seed27_eval_Env1,PPO_Seed27,PPO_Env1_Seed27,PPO,Env1,Env1,27,5,-2.4994,4999,...,NaN,ml_agents_inference,NaN,,MazeAgent/Assets/Models/PPO_Env1_Seed27.onnx,,2026-05-14T18:50:19,0,0,1


#### 16.3 PPO Env1 – Seed 42

Unity-Einstellungen:

- ONNX-Modell: `PPO_Env1_Seed42.onnx`
- Behavior Type: `Inference Only`
- EpisodeLogger Run Id: `PPO_Seed42_eval_Env1`
- Algorithm Name: `PPO`
- Environment Name: `Env1`
- Max Episodes: `50`

```bash
# Erwartete Datei nach dem Unity-Run:
MazeAgent/training/evaluation_logs/PPO_Seed42_eval_Env1.csv

# Zielordner für die finale Auswertung:
training/evaluation_results/raw/PPO_Seed42_eval_Env1.csv
```

In [43]:
ppo_seed42_env1_df = copy_ppo_eval_file("PPO_Seed42_eval_Env1")

PPO CSV normalisiert und kopiert:
Von: rl_navigation_project/MazeAgent/training/evaluation_logs/PPO_Seed42_eval_Env1.csv
Nach: rl_navigation_project/training/evaluation_results/raw/PPO_Seed42_eval_Env1.csv
Geladene Episoden: 50


,eval_run_id,analysis_run_id,original_run_id,algorithm,train_environment,eval_environment,seed,episode,episode_reward,episode_steps,...,deterministic,eval_mode,most_common_action,action_counts,model_path,info,evaluated_at,success_flag,wall_hit_flag,timeout_flag
0,PPO_Seed42_eval_Env1,PPO_Seed42,PPO_Env1_Seed42,PPO,Env1,Env1,42,1,-2.4994,4999,...,NaN,ml_agents_inference,NaN,,MazeAgent/Assets/Models/PPO_Env1_Seed42.onnx,,2026-05-14T18:50:22,0,0,1
1,PPO_Seed42_eval_Env1,PPO_Seed42,PPO_Env1_Seed42,PPO,Env1,Env1,42,2,-2.4994,4999,...,NaN,ml_agents_inference,NaN,,MazeAgent/Assets/Models/PPO_Env1_Seed42.onnx,,2026-05-14T18:50:22,0,0,1
2,PPO_Seed42_eval_Env1,PPO_Seed42,PPO_Env1_Seed42,PPO,Env1,Env1,42,3,-2.4994,4999,...,NaN,ml_agents_inference,NaN,,MazeAgent/Assets/Models/PPO_Env1_Seed42.onnx,,2026-05-14T18:50:22,0,0,1
3,PPO_Seed42_eval_Env1,PPO_Seed42,PPO_Env1_Seed42,PPO,Env1,Env1,42,4,-2.4994,4999,...,NaN,ml_agents_inference,NaN,,MazeAgent/Assets/Models/PPO_Env1_Seed42.onnx,,2026-05-14T18:50:22,0,0,1
4,PPO_Seed42_eval_Env1,PPO_Seed42,PPO_Env1_Seed42,PPO,Env1,Env1,42,5,-2.4994,4999,...,NaN,ml_agents_inference,NaN,,MazeAgent/Assets/Models/PPO_Env1_Seed42.onnx,,2026-05-14T18:50:22,0,0,1


#### 16.4 PPO Env1 – Seed 72

Unity-Einstellungen:

- ONNX-Modell: `PPO_Env1_Seed72.onnx`
- Behavior Type: `Inference Only`
- EpisodeLogger Run Id: `PPO_Seed72_eval_Env1`
- Algorithm Name: `PPO`
- Environment Name: `Env1`
- Max Episodes: `50`

```bash
# Erwartete Datei nach dem Unity-Run:
MazeAgent/training/evaluation_logs/PPO_Seed72_eval_Env1.csv

# Zielordner für die finale Auswertung:
training/evaluation_results/raw/PPO_Seed72_eval_Env1.csv
```

In [44]:
ppo_seed72_env1_df = copy_ppo_eval_file("PPO_Seed72_eval_Env1")

PPO CSV normalisiert und kopiert:
Von: rl_navigation_project/MazeAgent/training/evaluation_logs/PPO_Seed72_eval_Env1.csv
Nach: rl_navigation_project/training/evaluation_results/raw/PPO_Seed72_eval_Env1.csv
Geladene Episoden: 50


,eval_run_id,analysis_run_id,original_run_id,algorithm,train_environment,eval_environment,seed,episode,episode_reward,episode_steps,...,deterministic,eval_mode,most_common_action,action_counts,model_path,info,evaluated_at,success_flag,wall_hit_flag,timeout_flag
0,PPO_Seed72_eval_Env1,PPO_Seed72,PPO_Env1_Seed72,PPO,Env1,Env1,72,1,-2.4994,4999,...,NaN,ml_agents_inference,NaN,,MazeAgent/Assets/Models/PPO_Env1_Seed72.onnx,,2026-05-14T18:50:24,0,0,1
1,PPO_Seed72_eval_Env1,PPO_Seed72,PPO_Env1_Seed72,PPO,Env1,Env1,72,2,-2.4994,4999,...,NaN,ml_agents_inference,NaN,,MazeAgent/Assets/Models/PPO_Env1_Seed72.onnx,,2026-05-14T18:50:24,0,0,1
2,PPO_Seed72_eval_Env1,PPO_Seed72,PPO_Env1_Seed72,PPO,Env1,Env1,72,3,-2.4994,4999,...,NaN,ml_agents_inference,NaN,,MazeAgent/Assets/Models/PPO_Env1_Seed72.onnx,,2026-05-14T18:50:24,0,0,1
3,PPO_Seed72_eval_Env1,PPO_Seed72,PPO_Env1_Seed72,PPO,Env1,Env1,72,4,-2.4994,4999,...,NaN,ml_agents_inference,NaN,,MazeAgent/Assets/Models/PPO_Env1_Seed72.onnx,,2026-05-14T18:50:24,0,0,1
4,PPO_Seed72_eval_Env1,PPO_Seed72,PPO_Env1_Seed72,PPO,Env1,Env1,72,5,-2.4994,4999,...,NaN,ml_agents_inference,NaN,,MazeAgent/Assets/Models/PPO_Env1_Seed72.onnx,,2026-05-14T18:50:24,0,0,1


#### 16.5 PPO Env1 – Seed 100

Unity-Einstellungen:

- ONNX-Modell: `PPO_Env1_Seed100.onnx`
- Behavior Type: `Inference Only`
- EpisodeLogger Run Id: `PPO_Seed100_eval_Env1`
- Algorithm Name: `PPO`
- Environment Name: `Env1`
- Max Episodes: `50`

```bash
# Erwartete Datei nach dem Unity-Run:
MazeAgent/training/evaluation_logs/PPO_Seed100_eval_Env1.csv

# Zielordner für die finale Auswertung:
training/evaluation_results/raw/PPO_Seed100_eval_Env1.csv
```

In [45]:
ppo_seed100_env1_df = copy_ppo_eval_file("PPO_Seed100_eval_Env1")

PPO CSV normalisiert und kopiert:
Von: rl_navigation_project/MazeAgent/training/evaluation_logs/PPO_Seed100_eval_Env1.csv
Nach: rl_navigation_project/training/evaluation_results/raw/PPO_Seed100_eval_Env1.csv
Geladene Episoden: 50


,eval_run_id,analysis_run_id,original_run_id,algorithm,train_environment,eval_environment,seed,episode,episode_reward,episode_steps,...,deterministic,eval_mode,most_common_action,action_counts,model_path,info,evaluated_at,success_flag,wall_hit_flag,timeout_flag
0,PPO_Seed100_eval_Env1,PPO_Seed100,PPO_Env1_Seed100,PPO,Env1,Env1,100,1,-2.4994,4999,...,NaN,ml_agents_inference,NaN,,MazeAgent/Assets/Models/PPO_Env1_Seed100.onnx,,2026-05-14T18:50:27,0,0,1
1,PPO_Seed100_eval_Env1,PPO_Seed100,PPO_Env1_Seed100,PPO,Env1,Env1,100,2,-2.4994,4999,...,NaN,ml_agents_inference,NaN,,MazeAgent/Assets/Models/PPO_Env1_Seed100.onnx,,2026-05-14T18:50:27,0,0,1
2,PPO_Seed100_eval_Env1,PPO_Seed100,PPO_Env1_Seed100,PPO,Env1,Env1,100,3,-2.4994,4999,...,NaN,ml_agents_inference,NaN,,MazeAgent/Assets/Models/PPO_Env1_Seed100.onnx,,2026-05-14T18:50:27,0,0,1
3,PPO_Seed100_eval_Env1,PPO_Seed100,PPO_Env1_Seed100,PPO,Env1,Env1,100,4,-2.4994,4999,...,NaN,ml_agents_inference,NaN,,MazeAgent/Assets/Models/PPO_Env1_Seed100.onnx,,2026-05-14T18:50:27,0,0,1
4,PPO_Seed100_eval_Env1,PPO_Seed100,PPO_Env1_Seed100,PPO,Env1,Env1,100,5,-2.4994,4999,...,NaN,ml_agents_inference,NaN,,MazeAgent/Assets/Models/PPO_Env1_Seed100.onnx,,2026-05-14T18:50:27,0,0,1


---

### 17. Finale Evaluation-CSV-Dateien laden

Diese Funktion lädt alle finalen CSV-Dateien aus `training/evaluation_results/raw`.

Wichtig:
- Der Archivordner wird nicht geladen.
- Dadurch werden alte 200-Episoden-Runs nicht mit den finalen 50-Episoden-Runs vermischt.

In [17]:
def load_all_final_evaluation_csvs() -> pd.DataFrame:
    csv_files = sorted(EVALUATION_RAW_DIR.glob("*.csv"))

    frames = []

    for path in csv_files:
        try:
            df = pd.read_csv(path)
            df["source_file"] = show_project_path(path)
            frames.append(df)
        except Exception as exc:
            print(f"Konnte Datei nicht laden: {show_project_path(path)} | {exc}")

    if len(frames) == 0:
        print("Keine finalen Evaluation-CSV-Dateien gefunden.")
        return pd.DataFrame()

    combined_df = pd.concat(frames, ignore_index=True, sort=False)

    output_path = EVALUATION_TABLE_DIR / "all_final_evaluations_combined.csv"
    combined_df.to_csv(output_path, index=False, encoding="utf-8")

    print("Kombinierte finale Evaluation gespeichert:")
    print(show_project_path(output_path))

    return combined_df


all_eval_df = load_all_final_evaluation_csvs()
display(all_eval_df.head())

Kombinierte finale Evaluation gespeichert:
rl_navigation_project/training/evaluation_results/tables/all_final_evaluations_combined.csv


,eval_run_id,analysis_run_id,original_run_id,algorithm,train_environment,eval_environment,seed,episode,episode_reward,episode_steps,...,eval_mode,most_common_action,action_counts,model_path,info,evaluated_at,source_file,success_flag,wall_hit_flag,timeout_flag
0,A2C_Seed1_eval_Env1_deterministic,A2C_Seed1,A2C_Env1_Seed1,A2C,Env1,Env1,1,1,-2.5,5000,...,deterministic,0.0,"{5: 28, 6: 47, 1: 4, 7: 19, 0: 4902}",rl_navigation_project/training/models/final/A2...,NaN,2026-05-12T20:57:21,rl_navigation_project/training/evaluation_resu...,NaN,NaN,NaN
1,A2C_Seed1_eval_Env1_deterministic,A2C_Seed1,A2C_Env1_Seed1,A2C,Env1,Env1,1,2,-2.5,5000,...,deterministic,6.0,"{1: 5, 4: 4, 6: 2496, 3: 2495}",rl_navigation_project/training/models/final/A2...,NaN,2026-05-12T20:59:01,rl_navigation_project/training/evaluation_resu...,NaN,NaN,NaN
2,A2C_Seed1_eval_Env1_deterministic,A2C_Seed1,A2C_Env1_Seed1,A2C,Env1,Env1,1,3,-2.5,5000,...,deterministic,0.0,"{5: 28, 6: 47, 1: 4, 7: 19, 0: 4902}",rl_navigation_project/training/models/final/A2...,NaN,2026-05-12T21:00:42,rl_navigation_project/training/evaluation_resu...,NaN,NaN,NaN
3,A2C_Seed1_eval_Env1_deterministic,A2C_Seed1,A2C_Env1_Seed1,A2C,Env1,Env1,1,4,-2.5,5000,...,deterministic,0.0,"{5: 28, 6: 47, 1: 4, 7: 19, 0: 4902}",rl_navigation_project/training/models/final/A2...,NaN,2026-05-12T21:02:22,rl_navigation_project/training/evaluation_resu...,NaN,NaN,NaN
4,A2C_Seed1_eval_Env1_deterministic,A2C_Seed1,A2C_Env1_Seed1,A2C,Env1,Env1,1,5,-2.5,5000,...,deterministic,6.0,"{1: 5, 4: 4, 6: 2496, 3: 2495}",rl_navigation_project/training/models/final/A2...,NaN,2026-05-12T21:04:02,rl_navigation_project/training/evaluation_resu...,NaN,NaN,NaN


---

### 18. Evaluationsergebnisse zusammenfassen

Diese Zelle berechnet die wichtigsten Kennzahlen pro Run:

- Anzahl Episoden
- Mean Reward
- Mean Steps
- Goal Rate
- Timeout Rate
- Wall-like Failure Rate

Positive Rewards werden als Goal-Episoden interpretiert.  
Rewards kleiner oder gleich -5 werden als wall-like failures interpretiert.

In [18]:
def summarize_evaluation_results(eval_df: pd.DataFrame) -> pd.DataFrame:
    if eval_df.empty:
        return pd.DataFrame()

    eval_df = eval_df.copy()

    required_cols = [
        "algorithm",
        "analysis_run_id",
        "seed",
        "eval_environment",
        "episode_reward",
        "episode_steps",
    ]

    missing_cols = [col for col in required_cols if col not in eval_df.columns]

    if len(missing_cols) > 0:
        raise ValueError(f"Fehlende Spalten in eval_df: {missing_cols}")

    # Einheitliche Ergebnisflags erzeugen
    if "success_flag" not in eval_df.columns:
        eval_df["success_flag"] = (eval_df["episode_reward"] > 0).astype(int)
    else:
        eval_df["success_flag"] = eval_df["success_flag"].fillna(
            (eval_df["episode_reward"] > 0).astype(int)
        ).astype(int)

    if "wall_hit_flag" not in eval_df.columns:
        eval_df["wall_hit_flag"] = (eval_df["episode_reward"] <= -5).astype(int)
    else:
        eval_df["wall_hit_flag"] = eval_df["wall_hit_flag"].fillna(
            (eval_df["episode_reward"] <= -5).astype(int)
        ).astype(int)

    if "timeout_flag" not in eval_df.columns:
        eval_df["timeout_flag"] = (
            eval_df["episode_steps"] >= EVAL_MAX_STEPS_PER_EPISODE - 1
        ).astype(int)
    else:
        eval_df["timeout_flag"] = eval_df["timeout_flag"].fillna(
            (eval_df["episode_steps"] >= EVAL_MAX_STEPS_PER_EPISODE - 1).astype(int)
        ).astype(int)

    group_cols = ["algorithm", "analysis_run_id", "seed", "eval_environment"]

    if "eval_mode" in eval_df.columns:
        group_cols.append("eval_mode")
    elif "deterministic" in eval_df.columns:
        group_cols.append("deterministic")

    summary_df = (
        eval_df
        .groupby(group_cols)
        .agg(
            episodes=("episode", "count"),
            mean_reward=("episode_reward", "mean"),
            std_reward=("episode_reward", "std"),
            mean_steps=("episode_steps", "mean"),
            std_steps=("episode_steps", "std"),
            goals=("success_flag", "sum"),
            timeouts=("timeout_flag", "sum"),
            wall_like_failures=("wall_hit_flag", "sum"),
        )
        .reset_index()
    )

    summary_df["goal_rate"] = summary_df["goals"] / summary_df["episodes"]
    summary_df["timeout_rate"] = summary_df["timeouts"] / summary_df["episodes"]
    summary_df["wall_like_rate"] = summary_df["wall_like_failures"] / summary_df["episodes"]

    output_path = EVALUATION_TABLE_DIR / "evaluation_summary.csv"
    summary_df.to_csv(output_path, index=False, encoding="utf-8")

    print("Evaluation Summary gespeichert:")
    print(show_project_path(output_path))

    return summary_df

In [19]:
all_eval_df = load_all_final_evaluation_csvs()
evaluation_summary_df = summarize_evaluation_results(all_eval_df)

display(evaluation_summary_df)

Kombinierte finale Evaluation gespeichert:
rl_navigation_project/training/evaluation_results/tables/all_final_evaluations_combined.csv
Evaluation Summary gespeichert:
rl_navigation_project/training/evaluation_results/tables/evaluation_summary.csv


,algorithm,analysis_run_id,seed,eval_environment,eval_mode,episodes,mean_reward,std_reward,mean_steps,std_steps,goals,timeouts,wall_like_failures,goal_rate,timeout_rate,wall_like_rate
0,A2C,A2C_Seed1,1,Env1,deterministic,50,-2.608040,0.534659,4816.08,910.167956,0,48,2,0.00,0.96,0.04
1,A2C,A2C_Seed1,1,Env1,stochastic,50,-5.058170,1.384765,2116.34,1764.598476,0,10,40,0.00,0.20,0.80
2,A2C,A2C_Seed1,1,Env2,stochastic,50,-5.041960,0.075075,83.92,150.150107,0,0,50,0.00,0.00,1.00
3,A2C,A2C_Seed1,1,Env3,stochastic,50,-5.046110,0.055428,92.22,110.855171,0,0,50,0.00,0.00,1.00
4,A2C,A2C_Seed27,27,Env1,deterministic,50,-2.500000,0.000000,5000.00,0.000000,0,50,0,0.00,1.00,0.00
5,A2C,A2C_Seed27,27,Env1,stochastic,50,-5.329870,1.349085,2059.74,1842.593401,0,7,43,0.00,0.14,0.86
6,A2C,A2C_Seed27,27,Env2,deterministic,50,-4.855640,0.601216,311.28,1196.674935,0,3,47,0.00,0.06,0.94
7,A2C,A2C_Seed27,27,Env3,deterministic,50,-5.005300,0.004600,10.60,9.200710,0,0,50,0.00,0.00,1.00
8,A2C,A2C_Seed42,42,Env1,deterministic,50,-2.500000,0.000000,5000.00,0.000000,0,50,0,0.00,1.00,0.00
9,A2C,A2C_Seed42,42,Env1,stochastic,50,-5.156090,1.581048,2512.18,1912.708238,0,11,39,0.00,0.22,0.78


### 19. Beste Modelle für Transfer auswählen

Auf Basis der Env1-Evaluation werden die besten Modelle für die Transfer-Evaluation ausgewählt.

Kategorien:

- best_A2C_deterministic
- best_A2C_stochastic
- best_DQN_deterministic
- best_DQN_stochastic
- best_PPO

Auswahlkriterien in Reihenfolge:

1. höchste Goal Rate
2. höchster Mean Reward
3. niedrigste Timeout Rate
4. niedrigste Wall-like Failure Rate
5. niedrigste Mean Episode Length

Oder einfach gesagt: Goal Rate > Mean Reward > Timeout Rate > Wall-like Failure Rate > Mean Episode Length

Wenn ein Algorithmus keine Goals erreicht, kann trotzdem das relativ beste Modell anhand der übrigen Kriterien ausgewählt werden.

In [59]:
def select_best_model(
    summary_df: pd.DataFrame,
    algorithm: str,
    eval_mode: str | None = None,
    eval_environment: str = "Env1",
) -> pd.Series:
    df = summary_df[
        (summary_df["algorithm"] == algorithm) &
        (summary_df["eval_environment"] == eval_environment)
    ].copy()

    if eval_mode is not None and "eval_mode" in df.columns:
        df = df[df["eval_mode"] == eval_mode]

    if df.empty:
        raise ValueError(
            f"Keine Ergebnisse gefunden für algorithm={algorithm}, "
            f"eval_mode={eval_mode}, eval_environment={eval_environment}"
        )

    df = df.sort_values(
        by=["goal_rate", "mean_reward", "timeout_rate", "wall_like_rate", "mean_steps"],
        ascending=[False, False, True, True, True],
    )

    return df.iloc[0]


best_models = {}

for algorithm, eval_mode, label in [
    ("A2C", "deterministic", "best_A2C_deterministic"),
    ("A2C", "stochastic", "best_A2C_stochastic"),
    ("DQN", "deterministic", "best_DQN_deterministic"),
    ("DQN", "stochastic", "best_DQN_stochastic"),
]:
    try:
        best_models[label] = select_best_model(
            evaluation_summary_df,
            algorithm=algorithm,
            eval_mode=eval_mode,
            eval_environment="Env1",
        )
    except Exception as exc:
        print(f"{label} konnte nicht ausgewählt werden:", exc)

try:
    best_models["best_PPO"] = select_best_model(
        evaluation_summary_df,
        algorithm="PPO",
        eval_mode=None,
        eval_environment="Env1",
    )
except Exception as exc:
    print("best_PPO konnte nicht ausgewählt werden:", exc)

best_models_df = (
    pd.DataFrame(best_models)
    .T
    .reset_index()
    .rename(columns={"index": "best_model_label"})
)

best_models_path = EVALUATION_TABLE_DIR / "best_models_for_transfer.csv"
best_models_df.to_csv(best_models_path, index=False, encoding="utf-8")

print("Best Models gespeichert:")
print(show_project_path(best_models_path))

display(best_models_df)

Best Models gespeichert:
rl_navigation_project/training/evaluation_results/tables/best_models_for_transfer.csv


,best_model_label,algorithm,analysis_run_id,seed,eval_environment,eval_mode,episodes,mean_reward,std_reward,mean_steps,std_steps,goals,timeouts,wall_like_failures,goal_rate,timeout_rate,wall_like_rate
0,best_A2C_deterministic,A2C,A2C_Seed27,27,Env1,deterministic,50,-2.5,0.0,5000.0,0.0,0,50,0,0.0,1.0,0.0
1,best_A2C_stochastic,A2C,A2C_Seed1,1,Env1,stochastic,50,-5.05817,1.384765,2116.34,1764.598476,0,10,40,0.0,0.2,0.8
2,best_DQN_deterministic,DQN,DQN_v1_Seed1,1,Env1,deterministic,50,-2.97972,1.041913,4159.44,1829.650041,0,41,9,0.0,0.82,0.18
3,best_DQN_stochastic,DQN,DQN_v1_Seed27,27,Env1,stochastic,50,-2.43109,5.484505,1262.18,1865.061543,8,8,34,0.16,0.16,0.68
4,best_PPO,PPO,PPO_Seed1,1,Env1,ml_agents_inference,50,-2.4994,0.0,4999.0,0.0,0,50,0,0.0,1.0,0.0


### 20. Transfer-Evaluation vorbereiten

Die Transfer-Evaluation wird nur mit den besten Modellen aus Env1 durchgeführt.

Geplant:

- best_A2C_deterministic -> Env2 und Env3
- best_A2C_stochastic -> Env2 und Env3
- best_DQN_deterministic -> Env2 und Env3
- best_DQN_stochastic -> Env2 und Env3
- best_PPO -> Env2 und Env3 über Unity Inference

Damit wird nicht jede Seed-Kombination übertragen, sondern nur die bestperformenden Modelle je Kategorie.

In [50]:
def evaluate_best_sb3_model(
    best_model_label: str,
    eval_environment: str,
) -> pd.DataFrame:
    if best_model_label not in best_models:
        raise ValueError(f"{best_model_label} nicht in best_models gefunden.")

    row = best_models[best_model_label]

    algorithm = row["algorithm"]
    seed = int(row["seed"])

    deterministic = True

    if "eval_mode" in row.index:
        deterministic = row["eval_mode"] == "deterministic"

    print("Best Model:", best_model_label)
    print("Algorithm:", algorithm)
    print("Seed:", seed)
    print("Deterministic:", deterministic)
    print("Target Environment:", eval_environment)

    return evaluate_selected_sb3_model(
        algorithm=algorithm,
        seed=seed,
        eval_environment=eval_environment,
        deterministic=deterministic,
    )

---

### 21. SB3 Transfer-Runs

Diese Zellen führen die Transfer-Evaluation für die besten A2C- und DQN-Modelle aus. Das Modell wird hierbei mit seinem Trainingsseed identifiziert und daamit auch mit diesem Seed in Env2 und Env3 evaluiert

Vor jedem Run:
- passende Unity-Szene öffnen
- Env2 oder Env3 auswählen
- EpisodeLogger Run ID passend setzen
- Max Episodes: 50
- Python-Zelle starten
- Unity Play drücken

#### 21.1 Transfer best A2C deterministic to Env2

A2C_Seed27

In [51]:
transfer_best_a2c_deterministic_env2 = evaluate_best_sb3_model(
    best_model_label="best_A2C_deterministic",
    eval_environment="Env2",
)

Best Model: best_A2C_deterministic
Algorithm: A2C
Seed: 27
Deterministic: True
Target Environment: Env2
Evaluation Run: A2C_Seed27_eval_Env2_deterministic
Algorithmus: A2C
Seed: 27
Modell: rl_navigation_project/training/models/final/A2C_Env1_Seed27/final_model.zip
Evaluation Environment: Env2
Episoden: 50
Deterministic: True
Bitte jetzt in Unity die passende Szene öffnen und Play drücken.
Wenn Unity verbunden ist, startet die Evaluation automatisch.
Seeds gesetzt auf: 27
Seeds gesetzt auf: 27
Starte Verbindung zu Unity. Jetzt in Unity Play drücken...
[WARNING] The environment contains multiple observations. You must define allow_multiple_obs=True to receive them all. Otherwise, only the first visual observation (or vector observation ifthere are no visual observations) will be provided in the observation.
Unity Environment verbunden.
Observation Space: Box(-inf, inf, (48,), float32)
Action Space: Discrete(9)
A2C_Seed27_eval_Env2_deterministic: Episode 10/50 | Last10 Reward: -4.754 | La

#### 21.2 Transfer best A2C deterministic to Env3

A2C_Seed27

In [52]:
transfer_best_a2c_deterministic_env3 = evaluate_best_sb3_model(
    best_model_label="best_A2C_deterministic",
    eval_environment="Env3",
)

Best Model: best_A2C_deterministic
Algorithm: A2C
Seed: 27
Deterministic: True
Target Environment: Env3
Evaluation Run: A2C_Seed27_eval_Env3_deterministic
Algorithmus: A2C
Seed: 27
Modell: rl_navigation_project/training/models/final/A2C_Env1_Seed27/final_model.zip
Evaluation Environment: Env3
Episoden: 50
Deterministic: True
Bitte jetzt in Unity die passende Szene öffnen und Play drücken.
Wenn Unity verbunden ist, startet die Evaluation automatisch.
Seeds gesetzt auf: 27
Seeds gesetzt auf: 27
Starte Verbindung zu Unity. Jetzt in Unity Play drücken...
[WARNING] The environment contains multiple observations. You must define allow_multiple_obs=True to receive them all. Otherwise, only the first visual observation (or vector observation ifthere are no visual observations) will be provided in the observation.
Unity Environment verbunden.
Observation Space: Box(-inf, inf, (48,), float32)
Action Space: Discrete(9)
A2C_Seed27_eval_Env3_deterministic: Episode 10/50 | Last10 Reward: -5.007 | La

#### 21.3 Transfer best A2C stochastic to Env2

A2C_Seed1

In [53]:
transfer_best_a2c_stochastic_env2 = evaluate_best_sb3_model(
    best_model_label="best_A2C_stochastic",
    eval_environment="Env2",
)

Best Model: best_A2C_stochastic
Algorithm: A2C
Seed: 1
Deterministic: False
Target Environment: Env2
Evaluation Run: A2C_Seed1_eval_Env2_stochastic
Algorithmus: A2C
Seed: 1
Modell: rl_navigation_project/training/models/final/A2C_Env1_Seed1/final_model.zip
Evaluation Environment: Env2
Episoden: 50
Deterministic: False
Bitte jetzt in Unity die passende Szene öffnen und Play drücken.
Wenn Unity verbunden ist, startet die Evaluation automatisch.
Seeds gesetzt auf: 1
Seeds gesetzt auf: 1
Starte Verbindung zu Unity. Jetzt in Unity Play drücken...
[WARNING] The environment contains multiple observations. You must define allow_multiple_obs=True to receive them all. Otherwise, only the first visual observation (or vector observation ifthere are no visual observations) will be provided in the observation.
Unity Environment verbunden.
Observation Space: Box(-inf, inf, (48,), float32)
Action Space: Discrete(9)
A2C_Seed1_eval_Env2_stochastic: Episode 10/50 | Last10 Reward: -5.040 | Last10 Steps: 79

#### 21.4 Transfer best A2C stochastic to Env3

A2C_Seed1

In [54]:
transfer_best_a2c_stochastic_env3 = evaluate_best_sb3_model(
    best_model_label="best_A2C_stochastic",
    eval_environment="Env3",
)

Best Model: best_A2C_stochastic
Algorithm: A2C
Seed: 1
Deterministic: False
Target Environment: Env3
Evaluation Run: A2C_Seed1_eval_Env3_stochastic
Algorithmus: A2C
Seed: 1
Modell: rl_navigation_project/training/models/final/A2C_Env1_Seed1/final_model.zip
Evaluation Environment: Env3
Episoden: 50
Deterministic: False
Bitte jetzt in Unity die passende Szene öffnen und Play drücken.
Wenn Unity verbunden ist, startet die Evaluation automatisch.
Seeds gesetzt auf: 1
Seeds gesetzt auf: 1
Starte Verbindung zu Unity. Jetzt in Unity Play drücken...
[WARNING] The environment contains multiple observations. You must define allow_multiple_obs=True to receive them all. Otherwise, only the first visual observation (or vector observation ifthere are no visual observations) will be provided in the observation.
Unity Environment verbunden.
Observation Space: Box(-inf, inf, (48,), float32)
Action Space: Discrete(9)
A2C_Seed1_eval_Env3_stochastic: Episode 10/50 | Last10 Reward: -5.055 | Last10 Steps: 10

#### 21.5 Transfer best DQN deterministic to Env2

DQN_Seed1

In [55]:
transfer_best_dqn_deterministic_env2 = evaluate_best_sb3_model(
    best_model_label="best_DQN_deterministic",
    eval_environment="Env2",
)

Best Model: best_DQN_deterministic
Algorithm: DQN
Seed: 1
Deterministic: True
Target Environment: Env2
Evaluation Run: DQN_v1_Seed1_eval_Env2_deterministic
Algorithmus: DQN
Seed: 1
Modell: rl_navigation_project/training/models/final/DQN_Env1_Seed1/final_model.zip
Evaluation Environment: Env2
Episoden: 50
Deterministic: True
Bitte jetzt in Unity die passende Szene öffnen und Play drücken.
Wenn Unity verbunden ist, startet die Evaluation automatisch.
Seeds gesetzt auf: 1
Seeds gesetzt auf: 1
Starte Verbindung zu Unity. Jetzt in Unity Play drücken...
[WARNING] The environment contains multiple observations. You must define allow_multiple_obs=True to receive them all. Otherwise, only the first visual observation (or vector observation ifthere are no visual observations) will be provided in the observation.
Unity Environment verbunden.
Observation Space: Box(-inf, inf, (48,), float32)
Action Space: Discrete(9)
DQN_v1_Seed1_eval_Env2_deterministic: Episode 10/50 | Last10 Reward: -3.000 | Las

#### 21.6 Transfer best DQN deterministic to Env3

DQN_Seed1

In [56]:
transfer_best_dqn_deterministic_env3 = evaluate_best_sb3_model(
    best_model_label="best_DQN_deterministic",
    eval_environment="Env3",
)

Best Model: best_DQN_deterministic
Algorithm: DQN
Seed: 1
Deterministic: True
Target Environment: Env3
Evaluation Run: DQN_v1_Seed1_eval_Env3_deterministic
Algorithmus: DQN
Seed: 1
Modell: rl_navigation_project/training/models/final/DQN_Env1_Seed1/final_model.zip
Evaluation Environment: Env3
Episoden: 50
Deterministic: True
Bitte jetzt in Unity die passende Szene öffnen und Play drücken.
Wenn Unity verbunden ist, startet die Evaluation automatisch.
Seeds gesetzt auf: 1
Seeds gesetzt auf: 1
Starte Verbindung zu Unity. Jetzt in Unity Play drücken...
[WARNING] The environment contains multiple observations. You must define allow_multiple_obs=True to receive them all. Otherwise, only the first visual observation (or vector observation ifthere are no visual observations) will be provided in the observation.
Unity Environment verbunden.
Observation Space: Box(-inf, inf, (48,), float32)
Action Space: Discrete(9)
DQN_v1_Seed1_eval_Env3_deterministic: Episode 10/50 | Last10 Reward: -4.790 | Las

#### 21.7 Transfer best DQN stochastic to Env2

DQN_Seed27

In [57]:
transfer_best_dqn_stochastic_env2 = evaluate_best_sb3_model(
    best_model_label="best_DQN_stochastic",
    eval_environment="Env2",
)

Best Model: best_DQN_stochastic
Algorithm: DQN
Seed: 27
Deterministic: False
Target Environment: Env2
Evaluation Run: DQN_v1_Seed27_eval_Env2_stochastic
Algorithmus: DQN
Seed: 27
Modell: rl_navigation_project/training/models/final/DQN_Env1_Seed27/final_model.zip
Evaluation Environment: Env2
Episoden: 50
Deterministic: False
Bitte jetzt in Unity die passende Szene öffnen und Play drücken.
Wenn Unity verbunden ist, startet die Evaluation automatisch.
Seeds gesetzt auf: 27
Seeds gesetzt auf: 27
Starte Verbindung zu Unity. Jetzt in Unity Play drücken...
[WARNING] The environment contains multiple observations. You must define allow_multiple_obs=True to receive them all. Otherwise, only the first visual observation (or vector observation ifthere are no visual observations) will be provided in the observation.
Unity Environment verbunden.
Observation Space: Box(-inf, inf, (48,), float32)
Action Space: Discrete(9)
DQN_v1_Seed27_eval_Env2_stochastic: Episode 10/50 | Last10 Reward: -4.825 | Las

#### 21.8 Transfer best DQN stochastic to Env3

DQN_Seed27

In [58]:
transfer_best_dqn_stochastic_env3 = evaluate_best_sb3_model(
    best_model_label="best_DQN_stochastic",
    eval_environment="Env3",
)

Best Model: best_DQN_stochastic
Algorithm: DQN
Seed: 27
Deterministic: False
Target Environment: Env3
Evaluation Run: DQN_v1_Seed27_eval_Env3_stochastic
Algorithmus: DQN
Seed: 27
Modell: rl_navigation_project/training/models/final/DQN_Env1_Seed27/final_model.zip
Evaluation Environment: Env3
Episoden: 50
Deterministic: False
Bitte jetzt in Unity die passende Szene öffnen und Play drücken.
Wenn Unity verbunden ist, startet die Evaluation automatisch.
Seeds gesetzt auf: 27
Seeds gesetzt auf: 27
Starte Verbindung zu Unity. Jetzt in Unity Play drücken...
[WARNING] The environment contains multiple observations. You must define allow_multiple_obs=True to receive them all. Otherwise, only the first visual observation (or vector observation ifthere are no visual observations) will be provided in the observation.
Unity Environment verbunden.
Observation Space: Box(-inf, inf, (48,), float32)
Action Space: Discrete(9)
DQN_v1_Seed27_eval_Env3_stochastic: Episode 10/50 | Last10 Reward: -3.737 | Las

#### 21.9 Beste PPO-Modell für die Transfer-Evaluation

Da zwischen Seed 1 und Seed 100 kein leistungsbezogener Unterschied (siehe die definierten Auswahlkrietieren) festgestellt werden konnte, wurde als ein art technischer Tie-Breaker der niedrigere Seed (Seed 1) gewählt.

Diese Entscheidung ist keine inhaltliche Bevorzugung von Seed 1, sondern dient der eindeutigen und reproduzierbaren Auswahl bei vollständigem Gleichstand

In [20]:
ppo_best_candidates_df = evaluation_summary_df[
    (evaluation_summary_df["algorithm"] == "PPO") &
    (evaluation_summary_df["eval_environment"] == "Env1")
].copy()

ppo_best_candidates_df = ppo_best_candidates_df.sort_values(
    by=[
        "goal_rate",
        "mean_reward",
        "timeout_rate",
        "wall_like_rate",
        "mean_steps",
        "seed",
    ],
    ascending=[
        False,
        False,
        True,
        True,
        True,
        True,
    ],
)

ppo_best_candidates_path = EVALUATION_TABLE_DIR / "ppo_best_model_candidates.csv"
ppo_best_candidates_df.to_csv(ppo_best_candidates_path, index=False, encoding="utf-8")

print("PPO Best-Model-Kandidaten gespeichert:")
print(show_project_path(ppo_best_candidates_path))

display(ppo_best_candidates_df)

PPO Best-Model-Kandidaten gespeichert:
rl_navigation_project/training/evaluation_results/tables/ppo_best_model_candidates.csv


,algorithm,analysis_run_id,seed,eval_environment,eval_mode,episodes,mean_reward,std_reward,mean_steps,std_steps,goals,timeouts,wall_like_failures,goal_rate,timeout_rate,wall_like_rate
24,PPO,PPO_Seed1,1,Env1,ml_agents_inference,50,-2.499400,0.000000,4999.00,0.000000,0,50,0,0.0,1.00,0.00
25,PPO,PPO_Seed100,100,Env1,ml_agents_inference,50,-2.499400,0.000000,4999.00,0.000000,0,50,0,0.0,1.00,0.00
27,PPO,PPO_Seed42,42,Env1,ml_agents_inference,50,-2.554602,0.390337,4909.40,633.567676,0,49,1,0.0,0.98,0.02
28,PPO,PPO_Seed72,72,Env1,ml_agents_inference,50,-2.652856,0.613599,4705.90,1171.977559,0,47,3,0.0,0.94,0.06
26,PPO,PPO_Seed27,27,Env1,ml_agents_inference,50,-2.767226,0.937957,4734.64,986.006370,0,46,4,0.0,0.92,0.08


In [21]:
def copy_best_ppo_transfer_file(run_id: str) -> pd.DataFrame:
    source_path = PPO_UNITY_LOG_DIR / f"{run_id}.csv"
    target_path = EVALUATION_RAW_DIR / f"{run_id}.csv"

    if not source_path.exists():
        raise FileNotFoundError(
            f"PPO Transfer CSV nicht gefunden: {show_project_path(source_path)}"
        )

    raw_df = pd.read_csv(source_path)
    normalized_df = raw_df.copy()

    normalized_df = normalized_df.rename(columns={
        "run_id": "eval_run_id",
        "environment": "eval_environment",
        "total_reward": "episode_reward",
        "steps": "episode_steps",
    })

    normalized_df["eval_environment"] = normalized_df["eval_environment"].replace({
        "Env1_Simple": "Env1",
        "Env2_Simple": "Env2",
        "Env3_Simple": "Env3",
        "Env2_Complex": "Env2",
        "Env3_Warehouse": "Env3",
        "Env1": "Env1",
        "Env2": "Env2",
        "Env3": "Env3",
    })

    # best_PPO ist fest PPO Seed 1
    seed = 1

    normalized_df["eval_run_id"] = run_id
    normalized_df["analysis_run_id"] = "PPO_Seed1"
    normalized_df["original_run_id"] = "PPO_Env1_Seed1"
    normalized_df["algorithm"] = "PPO"
    normalized_df["train_environment"] = TRAIN_ENV_NAME
    normalized_df["seed"] = seed
    normalized_df["deterministic"] = np.nan
    normalized_df["eval_mode"] = "ml_agents_inference"
    normalized_df["timeout_by_python_limit"] = False
    normalized_df["most_common_action"] = np.nan
    normalized_df["action_counts"] = ""
    normalized_df["model_path"] = "MazeAgent/Assets/Models/PPO_Env1_Seed1.onnx"
    normalized_df["info"] = ""
    normalized_df["evaluated_at"] = pd.Timestamp.now().isoformat(timespec="seconds")
    normalized_df["done"] = True

    normalized_df["success_flag"] = normalized_df["success"].astype(int)
    normalized_df["wall_hit_flag"] = normalized_df["wall_hit"].astype(int)
    normalized_df["timeout_flag"] = normalized_df["timeout"].astype(int)

    final_cols = [
        "eval_run_id",
        "analysis_run_id",
        "original_run_id",
        "algorithm",
        "train_environment",
        "eval_environment",
        "seed",
        "episode",
        "episode_reward",
        "episode_steps",
        "done",
        "timeout_by_python_limit",
        "deterministic",
        "eval_mode",
        "most_common_action",
        "action_counts",
        "model_path",
        "info",
        "evaluated_at",
        "success_flag",
        "wall_hit_flag",
        "timeout_flag",
    ]

    normalized_df = normalized_df[final_cols]
    normalized_df.to_csv(target_path, index=False, encoding="utf-8")

    episodes = len(normalized_df)
    goals = normalized_df["success_flag"].sum()
    timeouts = normalized_df["timeout_flag"].sum()
    walls = normalized_df["wall_hit_flag"].sum()

    print("PPO Transfer CSV normalisiert und kopiert:")
    print("Von:", show_project_path(source_path))
    print("Nach:", show_project_path(target_path))
    print()
    print("Haupteckdaten:")
    print("Run ID:", run_id)
    print("Environment:", normalized_df["eval_environment"].iloc[0])
    print("Episoden:", episodes)
    print("Goals:", goals)
    print("Goal Rate:", goals / episodes)
    print("Timeouts:", timeouts)
    print("Timeout Rate:", timeouts / episodes)
    print("Wall Hits:", walls)
    print("Wall Rate:", walls / episodes)
    print("Mean Reward:", normalized_df["episode_reward"].mean())
    print("Mean Steps:", normalized_df["episode_steps"].mean())

    display(normalized_df.head())

    return normalized_df

#### 21.10 PPO Transfer-Evaluation über Unity Inference

Für die PPO-Transfer-Evaluation wird das ausgewählte Best Model `PPO Seed 1` verwendet.

Unity-Einstellungen für beide Transfer-Runs:

- MazeAgent -> Behavior Parameters:
  - Behavior Type: `Inference Only`
  - Model: `PPO_Env1_Seed1.onnx`
- EpisodeLogger:
  - Enable Logging: aktiv
  - Use Max Episodes: aktiv
  - Max Episodes: 50
  - Stop Play Mode: aktiv
  - Algorithm Name: `PPO`

Für Env2:

- Szene: Env2
- Run Id: `PPO_Seed1_eval_Env2`
- Environment Name: `Env2`

Für Env3:

- Szene: Env3
- Run Id: `PPO_Seed1_eval_Env3`
- Environment Name: `Env3`

Die erzeugten CSV-Dateien werden zunächst unter `MazeAgent/training/evaluation_logs` gespeichert und danach in den finalen Auswertungsordner `training/evaluation_results/raw` kopiert.

#### 21.11 PPO Transfer – Env2

Nach Abschluss des Unity-Runs `PPO_Seed1_eval_Env2` wird die CSV-Datei aus dem Unity-Logger-Ordner in den finalen Evaluation-Ordner kopiert und normalisiert.

```bash
# Erwartete Datei nach dem Unity-Run:
MazeAgent/training/evaluation_logs/best_PPO_eval_Env2.csv

# Zielordner für die finale Auswertung:
training/evaluation_results/raw/best_PPO_eval_Env2.csv
```

In [22]:
best_ppo_env3_df = copy_best_ppo_transfer_file("PPO_Seed1_eval_Env2")

PPO Transfer CSV normalisiert und kopiert:
Von: rl_navigation_project/MazeAgent/training/evaluation_logs/PPO_Seed1_eval_Env2.csv
Nach: rl_navigation_project/training/evaluation_results/raw/PPO_Seed1_eval_Env2.csv

Haupteckdaten:
Run ID: PPO_Seed1_eval_Env2
Environment: Env2
Episoden: 50
Goals: 0
Goal Rate: 0.0
Timeouts: 7
Timeout Rate: 0.14
Wall Hits: 43
Wall Rate: 0.86
Mean Reward: -4.652875999999999
Mean Steps: 705.78


,eval_run_id,analysis_run_id,original_run_id,algorithm,train_environment,eval_environment,seed,episode,episode_reward,episode_steps,...,deterministic,eval_mode,most_common_action,action_counts,model_path,info,evaluated_at,success_flag,wall_hit_flag,timeout_flag
0,PPO_Seed1_eval_Env2,PPO_Seed1,PPO_Env1_Seed1,PPO,Env1,Env2,1,1,-5.0030,6,...,NaN,ml_agents_inference,NaN,,MazeAgent/Assets/Models/PPO_Env1_Seed1.onnx,,2026-05-15T10:15:54,0,1,0
1,PPO_Seed1_eval_Env2,PPO_Seed1,PPO_Env1_Seed1,PPO,Env1,Env2,1,2,-5.0020,4,...,NaN,ml_agents_inference,NaN,,MazeAgent/Assets/Models/PPO_Env1_Seed1.onnx,,2026-05-15T10:15:54,0,1,0
2,PPO_Seed1_eval_Env2,PPO_Seed1,PPO_Env1_Seed1,PPO,Env1,Env2,1,3,-5.0030,6,...,NaN,ml_agents_inference,NaN,,MazeAgent/Assets/Models/PPO_Env1_Seed1.onnx,,2026-05-15T10:15:54,0,1,0
3,PPO_Seed1_eval_Env2,PPO_Seed1,PPO_Env1_Seed1,PPO,Env1,Env2,1,4,-5.0035,7,...,NaN,ml_agents_inference,NaN,,MazeAgent/Assets/Models/PPO_Env1_Seed1.onnx,,2026-05-15T10:15:54,0,1,0
4,PPO_Seed1_eval_Env2,PPO_Seed1,PPO_Env1_Seed1,PPO,Env1,Env2,1,5,-5.0030,6,...,NaN,ml_agents_inference,NaN,,MazeAgent/Assets/Models/PPO_Env1_Seed1.onnx,,2026-05-15T10:15:54,0,1,0


#### 21.12 PPO Transfer – Env3

Nach Abschluss des Unity-Runs `PPO_Seed1_eval_Env3` wird die CSV-Datei aus dem Unity-Logger-Ordner in den finalen Evaluation-Ordner kopiert und normalisiert.

```bash
# Erwartete Datei nach dem Unity-Run:
MazeAgent/training/evaluation_logs/best_PPO_eval_Env3.csv

# Zielordner für die finale Auswertung:
training/evaluation_results/raw/best_PPO_eval_Env3.csv
```

In [23]:
best_ppo_env3_df = copy_best_ppo_transfer_file("PPO_Seed1_eval_Env3")

PPO Transfer CSV normalisiert und kopiert:
Von: rl_navigation_project/MazeAgent/training/evaluation_logs/PPO_Seed1_eval_Env3.csv
Nach: rl_navigation_project/training/evaluation_results/raw/PPO_Seed1_eval_Env3.csv

Haupteckdaten:
Run ID: PPO_Seed1_eval_Env3
Environment: Env3
Episoden: 50
Goals: 0
Goal Rate: 0.0
Timeouts: 0
Timeout Rate: 0.0
Wall Hits: 50
Wall Rate: 1.0
Mean Reward: -5.086749999999999
Mean Steps: 173.5


,eval_run_id,analysis_run_id,original_run_id,algorithm,train_environment,eval_environment,seed,episode,episode_reward,episode_steps,...,deterministic,eval_mode,most_common_action,action_counts,model_path,info,evaluated_at,success_flag,wall_hit_flag,timeout_flag
0,PPO_Seed1_eval_Env3,PPO_Seed1,PPO_Env1_Seed1,PPO,Env1,Env3,1,1,-5.0030,6,...,NaN,ml_agents_inference,NaN,,MazeAgent/Assets/Models/PPO_Env1_Seed1.onnx,,2026-05-15T10:15:57,0,1,0
1,PPO_Seed1_eval_Env3,PPO_Seed1,PPO_Env1_Seed1,PPO,Env1,Env3,1,2,-5.0040,8,...,NaN,ml_agents_inference,NaN,,MazeAgent/Assets/Models/PPO_Env1_Seed1.onnx,,2026-05-15T10:15:57,0,1,0
2,PPO_Seed1_eval_Env3,PPO_Seed1,PPO_Env1_Seed1,PPO,Env1,Env3,1,3,-5.1015,203,...,NaN,ml_agents_inference,NaN,,MazeAgent/Assets/Models/PPO_Env1_Seed1.onnx,,2026-05-15T10:15:57,0,1,0
3,PPO_Seed1_eval_Env3,PPO_Seed1,PPO_Env1_Seed1,PPO,Env1,Env3,1,4,-5.0060,12,...,NaN,ml_agents_inference,NaN,,MazeAgent/Assets/Models/PPO_Env1_Seed1.onnx,,2026-05-15T10:15:57,0,1,0
4,PPO_Seed1_eval_Env3,PPO_Seed1,PPO_Env1_Seed1,PPO,Env1,Env3,1,5,-5.0150,30,...,NaN,ml_agents_inference,NaN,,MazeAgent/Assets/Models/PPO_Env1_Seed1.onnx,,2026-05-15T10:15:57,0,1,0


---

### 22. Abschluss der Evaluation

Nach allen finalen Evaluationen werden die CSV-Dateien erneut geladen und die Summary wird aktualisiert.

Wichtig:
- Archivierte 200-Episoden-Runs werden nicht geladen.
- Finale Vergleichsbasis sind 50 Episoden pro Run.
- A2C und DQN werden nach deterministic/stochastic getrennt ausgewertet.
- PPO wird separat als Unity-ML-Agents-Inference-Ergebnis dokumentiert.

In [24]:
all_eval_df = load_all_final_evaluation_csvs()
evaluation_summary_df = summarize_evaluation_results(all_eval_df)

display(evaluation_summary_df)

Kombinierte finale Evaluation gespeichert:
rl_navigation_project/training/evaluation_results/tables/all_final_evaluations_combined.csv
Evaluation Summary gespeichert:
rl_navigation_project/training/evaluation_results/tables/evaluation_summary.csv


,algorithm,analysis_run_id,seed,eval_environment,eval_mode,episodes,mean_reward,std_reward,mean_steps,std_steps,goals,timeouts,wall_like_failures,goal_rate,timeout_rate,wall_like_rate
0,A2C,A2C_Seed1,1,Env1,deterministic,50,-2.608040,0.534659,4816.08,910.167956,0,48,2,0.00,0.96,0.04
1,A2C,A2C_Seed1,1,Env1,stochastic,50,-5.058170,1.384765,2116.34,1764.598476,0,10,40,0.00,0.20,0.80
2,A2C,A2C_Seed1,1,Env2,stochastic,50,-5.041960,0.075075,83.92,150.150107,0,0,50,0.00,0.00,1.00
3,A2C,A2C_Seed1,1,Env3,stochastic,50,-5.046110,0.055428,92.22,110.855171,0,0,50,0.00,0.00,1.00
4,A2C,A2C_Seed27,27,Env1,deterministic,50,-2.500000,0.000000,5000.00,0.000000,0,50,0,0.00,1.00,0.00
5,A2C,A2C_Seed27,27,Env1,stochastic,50,-5.329870,1.349085,2059.74,1842.593401,0,7,43,0.00,0.14,0.86
6,A2C,A2C_Seed27,27,Env2,deterministic,50,-4.855640,0.601216,311.28,1196.674935,0,3,47,0.00,0.06,0.94
7,A2C,A2C_Seed27,27,Env3,deterministic,50,-5.005300,0.004600,10.60,9.200710,0,0,50,0.00,0.00,1.00
8,A2C,A2C_Seed42,42,Env1,deterministic,50,-2.500000,0.000000,5000.00,0.000000,0,50,0,0.00,1.00,0.00
9,A2C,A2C_Seed42,42,Env1,stochastic,50,-5.156090,1.581048,2512.18,1912.708238,0,11,39,0.00,0.22,0.78


---

### 23. Nächster Schritt

Die Visualisierungen werden im finalen Ergebnisnotebook erstellt.

Geplante Plots:

1. Goal Rate nach Algorithmus, Seed und Evaluationsmodus in Env1
2. Mean Reward nach Algorithmus, Seed und Evaluationsmodus in Env1
3. Timeout Rate nach Algorithmus, Seed und Evaluationsmodus in Env1
4. Wall-like Failure Rate nach Algorithmus, Seed und Evaluationsmodus in Env1
5. Deterministic vs. stochastic Vergleich für A2C und DQN
6. Transfervergleich der Best Models auf Env1, Env2 und Env3
7. Random Baseline vs. trainierte Modelle